# Graph Cold-OD Ablation v2: Node/Edge Causality Controls

This notebook runs a stricter Cold-OD graph ablation suite for FREIGHT-MNet. The previous graph ablation showed that HGT/GraphSAGE-style models strongly outperform non-graph baselines in Cold-OD prediction, but self-loop, shuffled-edge, and randomized-edge variants remained surprisingly strong.

This v2 notebook separates the following sources of performance:

1. **Node-feature representation**: origin/destination FAF node features without message passing.
2. **Zone identity memorization**: learned origin/destination FAF embeddings without graph edges.
3. **Degree and zone-code shortcut controls**: remove zone numeric and train-degree node features.
4. **Topology-only signal**: constant node features with graph edges.
5. **Train-OD graph contribution**: full graph with train-OD edges removed, plus train-OD-only reference.
6. **Demand-mode contribution**: truck-only, rail-only, multimodal-only, and pairwise demand-mode graph variants.
7. **Perturbation robustness**: five independent shuffled-edge and randomized-edge-type controls.

All evaluations use the existing Cold-OD split: training OD pairs are disjoint from validation/test OD pairs. The main outputs are test-only leaderboards, test-only segment summaries, paired comparisons, and five-seed summaries.


## 1. Environment setup and imports

This cell imports only the packages required for this notebook. It intentionally avoids `DataFrame.to_markdown()` and the optional `tabulate` package, because previous notebook runs showed that `tabulate` may not be installed in the active Jupyter kernel.

In [1]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

# Avoid optional pandas acceleration paths that can trigger binary-extension issues.
os.environ.setdefault("NUMEXPR_MAX_THREADS", "1")

import numpy as np
import pandas as pd

pd.set_option("compute.use_numexpr", False)
pd.set_option("compute.use_bottleneck", False)

import torch
from torch import nn
import torch.nn.functional as F

try:
    from torch_geometric.nn import HGTConv, SAGEConv
    TORCH_GEOMETRIC_AVAILABLE = True
    TORCH_GEOMETRIC_IMPORT_ERROR = None
except Exception as exc:  # pragma: no cover
    TORCH_GEOMETRIC_AVAILABLE = False
    TORCH_GEOMETRIC_IMPORT_ERROR = repr(exc)

print("Python executable:", os.sys.executable)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
print("PyTorch Geometric available:", TORCH_GEOMETRIC_AVAILABLE)
if not TORCH_GEOMETRIC_AVAILABLE:
    print("PyG import error:", TORCH_GEOMETRIC_IMPORT_ERROR)
    raise RuntimeError("torch_geometric is required for this graph ablation notebook.")

Python executable: E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe
NumPy: 2.4.5
Pandas: 2.3.3
Torch: 2.12.0+cu126
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4050 Laptop GPU
PyTorch Geometric available: True


## 2. Experiment configuration

The configuration below controls data paths, Cold-OD split reuse, ablation names, seeds, optimization, and output paths. By default it runs the full requested ablation list over five seeds.

If runtime is too long, set `enabled_ablation_names` to a smaller tuple for a smoke test, then restore the full list for the final run.

In [2]:
@dataclass
class ExperimentConfig:
    """Configuration for the stricter Cold-OD graph causality ablation experiment."""

    # Project paths.
    data_root: Path = Path(r"E:\NetworkOptimization\pythonProject1\Data")
    scope: str = "east_plus_gulf"
    run_name: str = "graph_cold_od_ablation_v2_controls_notebook"

    supervised_relative_path: str = (
        r"08_processed\model_ready\freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet"
    )
    manifest_relative_path: str = (
        r"08_processed\model_ready\_metadata\freight_mnet_supervised_feature_manifest.json"
    )

    # Previous Cold-OD non-graph baseline run. This is used to reuse the exact Cold-OD split
    # and to add non-graph baselines to the combined leaderboard.
    cold_baseline_run_name: str = "cold_od_split_baselines_v1_notebook"
    cold_baseline_predictions_name: str = "predictions_cold_od_val_test.parquet"
    cold_baseline_all_splits_name: str = "predictions_cold_od_all_splits.parquet"

    # Fallback split settings used only if previous Cold-OD prediction files are unavailable.
    split_seed: int = 42
    val_pair_fraction: float = 0.10
    test_pair_fraction: float = 0.10
    train_years: Tuple[int, ...] = (2018, 2019, 2020, 2021, 2022)
    val_year: int = 2023
    test_year: int = 2024

    # Model-training seeds.
    seeds: Tuple[int, ...] = (7, 42, 123, 2026, 535)

    # Independent graph-perturbation seeds for repeated shuffled/randomized controls.
    graph_shuffle_seeds: Tuple[int, ...] = (101, 202, 303, 404, 505)
    edge_type_randomization_seeds: Tuple[int, ...] = (111, 222, 333, 444, 555)

    # Requested ablation list.
    # The list is intentionally broad. For a smoke test, temporarily reduce it to a few entries.
    enabled_ablation_names: Tuple[str, ...] = (
        # Reference graph and model-capacity controls.
        "hgt_full_faf_zone_graph",
        "graphsage_same_edges",
        "mlp_same_features_no_graph",
        "node_features_only_edge_decoder",
        "node_id_embedding_only",
        "hgt_self_loop_only",
        # Node-feature shortcut controls.
        "hgt_no_zone_numeric_no_degree_features",
        "hgt_constant_node_features_full_graph",
        # Train-OD and demand-edge controls.
        "hgt_train_od_only",
        "hgt_train_od_edges_removed_but_demand_kept",
        "hgt_demand_edges_only",
        # Demand-mode split controls.
        "hgt_truck_demand_only",
        "hgt_rail_demand_only",
        "hgt_multimodal_demand_only",
        "hgt_truck_plus_rail_demand",
        "hgt_truck_plus_multimodal_demand",
        "hgt_rail_plus_multimodal_demand",
        # Multiple shuffled-edge controls.
        "hgt_shuffled_graph_edges_s1",
        "hgt_shuffled_graph_edges_s2",
        "hgt_shuffled_graph_edges_s3",
        "hgt_shuffled_graph_edges_s4",
        "hgt_shuffled_graph_edges_s5",
        # Multiple randomized-edge-type controls.
        "hgt_randomized_edge_types_s1",
        "hgt_randomized_edge_types_s2",
        "hgt_randomized_edge_types_s3",
        "hgt_randomized_edge_types_s4",
        "hgt_randomized_edge_types_s5",
    )

    # Optimization settings.
    max_epochs: int = 500
    patience: int = 70
    batch_size: int = 4096
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    grad_clip_norm: float = 5.0
    lambda_iqr: float = 0.10
    eval_every: int = 1

    # Model settings.
    hidden_dim: int = 128
    graph_layers: int = 2
    hgt_heads: int = 4
    dropout: float = 0.10
    edge_hidden_dim: int = 128
    year_embedding_dim: int = 16

    # Training weights.
    use_sample_weight: bool = True
    weight_column: str = "obs_weight_sum"
    weight_clip_min: float = 0.05
    weight_clip_max: float = 20.0

    # Artifact controls.
    save_model_checkpoints: bool = True
    build_seed_ensembles: bool = True
    make_plots: bool = True
    overwrite: bool = True


cfg = ExperimentConfig()

project_root = cfg.data_root
supervised_path = project_root / cfg.supervised_relative_path
manifest_path = project_root / cfg.manifest_relative_path
cold_baseline_dir = project_root / "10_experiments" / cfg.cold_baseline_run_name / cfg.scope
cold_baseline_predictions_path = cold_baseline_dir / cfg.cold_baseline_predictions_name
cold_baseline_all_splits_path = cold_baseline_dir / cfg.cold_baseline_all_splits_name

output_dir = project_root / "10_experiments" / cfg.run_name / cfg.scope
models_dir = output_dir / "models"
tables_dir = output_dir / "tables"
plots_dir = output_dir / "plots"
reports_dir = output_dir / "reports"

for directory in [output_dir, models_dir, tables_dir, plots_dir, reports_dir]:
    directory.mkdir(parents=True, exist_ok=True)

print("Supervised table:", supervised_path)
print("Feature manifest:", manifest_path)
print("Previous Cold-OD predictions:", cold_baseline_predictions_path)
print("Output directory:", output_dir)
print("Enabled ablations:", cfg.enabled_ablation_names)
print("Training seeds:", cfg.seeds)
print("Shuffle seeds:", cfg.graph_shuffle_seeds)
print("Randomized-edge-type seeds:", cfg.edge_type_randomization_seeds)


Supervised table: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet
Feature manifest: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\_metadata\freight_mnet_supervised_feature_manifest.json
Previous Cold-OD predictions: E:\NetworkOptimization\pythonProject1\Data\10_experiments\cold_od_split_baselines_v1_notebook\east_plus_gulf\predictions_cold_od_val_test.parquet
Output directory: E:\NetworkOptimization\pythonProject1\Data\10_experiments\graph_cold_od_ablation_v2_controls_notebook\east_plus_gulf
Enabled ablations: ('hgt_full_faf_zone_graph', 'graphsage_same_edges', 'mlp_same_features_no_graph', 'node_features_only_edge_decoder', 'node_id_embedding_only', 'hgt_self_loop_only', 'hgt_no_zone_numeric_no_degree_features', 'hgt_constant_node_features_full_graph', 'hgt_train_od_only', 'hgt_train_od_edges_removed_but_demand_kept', 'hgt_demand_edges_only', 'hgt_truck_demand_only', 'hgt_rail_dema

## 3. Constants and reproducibility utilities

These helper functions standardize FAF zone IDs, locate schema columns, enforce deterministic seeds, and prevent accidental feature leakage.

In [3]:
LABEL_COLUMNS = ["truck_q25", "truck_q50", "truck_q75"]
QUANTILE_NAMES = ["q25", "q50", "q75"]
TAU_VALUES = np.array([0.25, 0.50, 0.75], dtype=np.float32)
TAUS_TENSOR = torch.tensor(TAU_VALUES, dtype=torch.float32)
CHECKPOINT_METRICS = ["best_val_pinball", "best_val_q75", "best_val_iqr"]

DEFAULT_ORIGIN_CANDIDATES = ["faf_orig", "orig_faf", "origin_faf", "origin", "orig"]
DEFAULT_DEST_CANDIDATES = ["faf_dest", "dest_faf", "destination_faf", "destination", "dest"]
DEFAULT_YEAR_CANDIDATES = ["year", "Year"]


def set_global_seed(seed: int) -> None:
    """Set Python, NumPy, and PyTorch seeds."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False


def format_faf_zone(value: Any) -> str:
    """Convert FAF zone identifiers to stable three-digit strings when possible."""
    if pd.isna(value):
        return "NA"
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    if text.isdigit():
        return f"{int(text):03d}"
    return text


def find_first_existing_column(df: pd.DataFrame, candidates: Sequence[str], role: str) -> str:
    """Return the first candidate column found in a dataframe."""
    for column in candidates:
        if column in df.columns:
            return column
    raise KeyError(f"Could not find {role} column. Candidates: {candidates}")


def ensure_columns(df: pd.DataFrame, columns: Sequence[str], context: str) -> None:
    """Raise a clear error if required columns are missing."""
    missing = [column for column in columns if column not in df.columns]
    if missing:
        raise KeyError(f"Missing columns in {context}: {missing}")


def stable_hash_to_int(text: str, seed: int = 0) -> int:
    """Convert a string into a deterministic integer hash."""
    payload = f"{seed}::{text}".encode("utf-8")
    return int(hashlib.md5(payload).hexdigest()[:12], 16)


def to_float32_array(frame: pd.DataFrame, columns: Sequence[str]) -> np.ndarray:
    """Convert selected columns to a float32 NumPy array."""
    return frame[list(columns)].astype(float).to_numpy(dtype=np.float32)

## 4. Load supervised table and feature manifest

The supervised table contains the FAF OD-year labels and the 64 manifest features. The manifest features are used for edge-level OD features, while FMI diagnostics such as `obs_weight_sum` and `n_fmi_county_pairs` are used only for weighting or diagnostics.

In [4]:
def load_supervised_table(path: Path) -> Tuple[pd.DataFrame, str, str, str]:
    """Load the supervised parquet table and normalize key columns."""
    if not path.exists():
        raise FileNotFoundError(f"Supervised parquet not found: {path}")

    df_local = pd.read_parquet(path)
    origin_col_local = find_first_existing_column(df_local, DEFAULT_ORIGIN_CANDIDATES, "origin")
    dest_col_local = find_first_existing_column(df_local, DEFAULT_DEST_CANDIDATES, "destination")
    year_col_local = find_first_existing_column(df_local, DEFAULT_YEAR_CANDIDATES, "year")

    ensure_columns(df_local, LABEL_COLUMNS, "supervised table")
    df_local = df_local.copy()
    df_local[origin_col_local] = df_local[origin_col_local].map(format_faf_zone)
    df_local[dest_col_local] = df_local[dest_col_local].map(format_faf_zone)
    df_local[year_col_local] = df_local[year_col_local].astype(int)
    df_local["od_pair"] = df_local[origin_col_local].astype(str) + "->" + df_local[dest_col_local].astype(str)
    df_local["row_id"] = np.arange(len(df_local), dtype=np.int64)

    for column in LABEL_COLUMNS:
        df_local[column] = pd.to_numeric(df_local[column], errors="coerce").astype(float)
    if df_local[LABEL_COLUMNS].isna().any().any():
        raise ValueError("Label columns contain NaN values after numeric conversion.")
    return df_local, origin_col_local, dest_col_local, year_col_local


def load_feature_manifest(path: Path) -> List[str]:
    """Load predictive feature columns from the manifest JSON."""
    if not path.exists():
        raise FileNotFoundError(f"Feature manifest not found: {path}")
    with path.open("r", encoding="utf-8") as file:
        manifest = json.load(file)
    candidates = [manifest.get("feature_columns"), manifest.get("manifest_feature_columns"), manifest.get("features")]
    for candidate in candidates:
        if isinstance(candidate, list) and candidate:
            return [str(column) for column in candidate]
    raise KeyError("Could not find a feature column list in the feature manifest.")


def leakage_guard(feature_columns: Sequence[str]) -> None:
    """Ensure predictive features do not include labels or FMI aggregation diagnostics."""
    forbidden_exact = set(LABEL_COLUMNS) | {
        "obs_weight_sum",
        "n_fmi_county_pairs",
        "input_q25_min",
        "input_q25_max",
        "input_q50_min",
        "input_q50_max",
        "input_q75_min",
        "input_q75_max",
    }
    forbidden_substrings = ["truck_q25", "truck_q50", "truck_q75", "fmi", "movement", "crossing"]
    violations = []
    for column in feature_columns:
        lower = column.lower()
        if column in forbidden_exact or any(token in lower for token in forbidden_substrings):
            violations.append(column)
    if violations:
        raise ValueError(f"Potential leakage features found: {violations}")


df, origin_col, dest_col, year_col = load_supervised_table(supervised_path)
manifest_feature_columns = load_feature_manifest(manifest_path)
leakage_guard(manifest_feature_columns)
ensure_columns(df, manifest_feature_columns, "supervised table manifest features")

for column in manifest_feature_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce").astype(float)

print("Data shape:", df.shape)
print("Origin/destination/year columns:", origin_col, dest_col, year_col)
print("Manifest feature count:", len(manifest_feature_columns))
print("Year counts:")
print(df[year_col].value_counts().sort_index())

Data shape: (73972, 94)
Origin/destination/year columns: faf_orig faf_dest year
Manifest feature count: 64
Year counts:
year
2018     9936
2019    10704
2020    10761
2021    10721
2022    10651
2023    10625
2024    10574
Name: count, dtype: int64


## 5. Label monotonicity and schema checks

The target quantiles should already satisfy `q25 <= q50 <= q75`. This cell verifies that assumption before training graph models.

In [5]:
def check_label_monotonicity(frame: pd.DataFrame) -> None:
    """Verify q25 <= q50 <= q75 for all supervised rows."""
    bad_q50 = frame["truck_q25"] > frame["truck_q50"]
    bad_q75 = frame["truck_q50"] > frame["truck_q75"]
    n_bad = int(bad_q50.sum() + bad_q75.sum())
    if n_bad:
        raise ValueError(f"Found {n_bad} monotonicity violations in labels.")


check_label_monotonicity(df)
print("Label monotonicity check passed.")

Label monotonicity check passed.


## 6. Load or reconstruct the Cold-OD split

This notebook reuses the previous Cold-OD split when available, so the ablation results are directly comparable to the Cold-OD non-graph baselines and the GraphSAGE/HGT v2 results.

If the previous prediction file is unavailable, the cell constructs a deterministic fallback Cold-OD split by hashing OD pairs.

In [6]:
def normalize_prediction_key_columns(frame: pd.DataFrame, origin_name: str, dest_name: str, year_name: str) -> pd.DataFrame:
    """Normalize key columns in a prediction dataframe."""
    result = frame.copy()
    if origin_name not in result.columns:
        result_origin = find_first_existing_column(result, DEFAULT_ORIGIN_CANDIDATES, "prediction origin")
        result = result.rename(columns={result_origin: origin_name})
    if dest_name not in result.columns:
        result_dest = find_first_existing_column(result, DEFAULT_DEST_CANDIDATES, "prediction destination")
        result = result.rename(columns={result_dest: dest_name})
    if year_name not in result.columns:
        result_year = find_first_existing_column(result, DEFAULT_YEAR_CANDIDATES, "prediction year")
        result = result.rename(columns={result_year: year_name})

    result[origin_name] = result[origin_name].map(format_faf_zone)
    result[dest_name] = result[dest_name].map(format_faf_zone)
    result[year_name] = result[year_name].astype(int)
    return result


def load_cold_split_from_predictions(path: Path, frame: pd.DataFrame) -> Optional[pd.Series]:
    """Infer cold_split from a previous all-splits prediction parquet file."""
    if not path.exists():
        return None
    predictions = pd.read_parquet(path)
    predictions = normalize_prediction_key_columns(predictions, origin_col, dest_col, year_col)
    if "split" not in predictions.columns:
        return None

    key_cols = [origin_col, dest_col, year_col]
    split_map = predictions[key_cols + ["split"]].drop_duplicates().copy()
    duplicate_keys = split_map.duplicated(key_cols, keep=False)
    if duplicate_keys.any():
        # Drop exact duplicates and keep one split per key. Multiple models produce the same key/split rows.
        split_map = split_map.drop_duplicates(key_cols)

    merged = frame[key_cols].merge(split_map, on=key_cols, how="left")
    cold_split = merged["split"].fillna("unused").astype(str)
    return cold_split


def build_deterministic_cold_od_split(frame: pd.DataFrame) -> pd.Series:
    """Build a deterministic fallback Cold-OD split by hashing OD pairs."""
    od_pairs = sorted(frame["od_pair"].unique().tolist())
    hashed = sorted((stable_hash_to_int(pair, cfg.split_seed), pair) for pair in od_pairs)
    n_pairs = len(hashed)
    n_val = int(round(n_pairs * cfg.val_pair_fraction))
    n_test = int(round(n_pairs * cfg.test_pair_fraction))
    val_pairs = {pair for _, pair in hashed[:n_val]}
    test_pairs = {pair for _, pair in hashed[n_val:n_val + n_test]}

    split = pd.Series("unused", index=frame.index, dtype="object")
    train_mask = frame[year_col].isin(cfg.train_years) & ~frame["od_pair"].isin(val_pairs | test_pairs)
    val_mask = frame[year_col].eq(cfg.val_year) & frame["od_pair"].isin(val_pairs)
    test_mask = frame[year_col].eq(cfg.test_year) & frame["od_pair"].isin(test_pairs)
    split.loc[train_mask] = "train"
    split.loc[val_mask] = "val"
    split.loc[test_mask] = "test"
    return split


def assert_cold_od_integrity(frame: pd.DataFrame) -> None:
    """Ensure validation and test OD pairs are not present in training OD pairs."""
    train_pairs = set(frame.loc[frame["cold_split"].eq("train"), "od_pair"])
    val_pairs = set(frame.loc[frame["cold_split"].eq("val"), "od_pair"])
    test_pairs = set(frame.loc[frame["cold_split"].eq("test"), "od_pair"])
    if train_pairs & val_pairs:
        raise AssertionError("Cold-OD leakage: training and validation OD pairs overlap.")
    if train_pairs & test_pairs:
        raise AssertionError("Cold-OD leakage: training and test OD pairs overlap.")
    if val_pairs & test_pairs:
        raise AssertionError("Cold-OD leakage: validation and test OD pairs overlap.")


cold_split = load_cold_split_from_predictions(cold_baseline_all_splits_path, df)
if cold_split is None:
    print("Previous Cold-OD all-splits predictions not found. Building deterministic fallback split.")
    cold_split = build_deterministic_cold_od_split(df)

df["cold_split"] = cold_split.astype(str)
assert_cold_od_integrity(df)

split_summary = (
    df.groupby("cold_split")
    .agg(
        n_rows=("cold_split", "size"),
        n_od_pairs=("od_pair", "nunique"),
        min_year=(year_col, "min"),
        max_year=(year_col, "max"),
    )
    .reset_index()
)
print(split_summary)

  cold_split  n_rows  n_od_pairs  min_year  max_year
0       test    1057        1057      2024      2024
1      train   42849        8748      2018      2022
2     unused   29109       10631      2018      2024
3        val     957         957      2023      2023


## 7. Sample weights

`obs_weight_sum` is used as a loss/metric weight, not as a predictive feature. The weights are log-transformed, clipped, and normalized by the training split mean.

In [7]:
def build_sample_weights(frame: pd.DataFrame, split_col: str, config: ExperimentConfig) -> np.ndarray:
    """Build normalized sample weights from obs_weight_sum when available."""
    if not config.use_sample_weight or config.weight_column not in frame.columns:
        return np.ones(len(frame), dtype=np.float32)
    raw = pd.to_numeric(frame[config.weight_column], errors="coerce").fillna(0.0).to_numpy(dtype=np.float32)
    weights = np.log1p(np.maximum(raw, 0.0)).astype(np.float32)
    train_mask = frame[split_col].eq("train").to_numpy()
    train_mean = float(np.mean(weights[train_mask])) if train_mask.any() else float(np.mean(weights))
    if train_mean <= 0 or not np.isfinite(train_mean):
        train_mean = 1.0
    weights = weights / train_mean
    weights = np.clip(weights, config.weight_clip_min, config.weight_clip_max).astype(np.float32)
    return weights


df["sample_weight"] = build_sample_weights(df, "cold_split", cfg)
print(df.groupby("cold_split")["sample_weight"].describe())

              count      mean       std       min       25%       50%  \
cold_split                                                              
test         1057.0  0.960896  0.324708  0.124688  0.735303  0.955175   
train       42849.0  1.000000  0.339350  0.124688  0.758963  1.005018   
unused      29109.0  0.988679  0.330799  0.124688  0.752114  0.986080   
val           957.0  0.987425  0.338340  0.124688  0.769588  0.988232   

                 75%       max  
cold_split                      
test        1.184582  2.004303  
train       1.244374  2.062156  
unused      1.226660  2.100173  
val         1.236716  1.833871  


## 8. Cold-OD fallback priors

Cold-OD test OD pairs have no exact OD-pair history. Therefore the graph and MLP residual models use a fallback prior based only on training-row global, origin-level, and destination-level medians.

For training rows, origin/destination fallback priors are built using an OD-pair-excluded calculation to reduce self-leakage.

In [8]:
COLD_PRIOR_COLUMNS = [
    "cold_prior_truck_q25",
    "cold_prior_truck_q50",
    "cold_prior_truck_q75",
    "cold_prior_iqr",
    "cold_prior_q75_q50_gap",
    "cold_prior_q50_q25_gap",
    "cold_has_origin_prior",
    "cold_has_dest_prior",
    "cold_has_any_zone_prior",
]


def median_vector(frame: pd.DataFrame) -> np.ndarray:
    """Return q25/q50/q75 medians as a float32 vector."""
    return frame[LABEL_COLUMNS].median().to_numpy(dtype=np.float32)


def build_group_median_map(train_frame: pd.DataFrame, group_col: str) -> Dict[str, np.ndarray]:
    """Build group-level label median vectors from training rows."""
    mapping: Dict[str, np.ndarray] = {}
    for key, group in train_frame.groupby(group_col):
        mapping[str(key)] = median_vector(group)
    return mapping


def blend_origin_destination_prior(origin_vec: Optional[np.ndarray], dest_vec: Optional[np.ndarray], global_vec: np.ndarray) -> Tuple[np.ndarray, bool, bool]:
    """Blend origin and destination prior vectors with global fallback."""
    has_origin = origin_vec is not None
    has_dest = dest_vec is not None
    if has_origin and has_dest:
        prior = 0.5 * origin_vec + 0.5 * dest_vec
    elif has_origin:
        prior = origin_vec
    elif has_dest:
        prior = dest_vec
    else:
        prior = global_vec
    prior = np.asarray(prior, dtype=np.float32)
    prior = np.sort(prior)
    return prior, has_origin, has_dest


def build_cold_priors(frame: pd.DataFrame) -> pd.DataFrame:
    """Attach Cold-OD fallback prior columns to the full dataframe."""
    result = frame.copy()
    train_frame = result[result["cold_split"].eq("train")].copy()
    global_vec = median_vector(train_frame)
    origin_map = build_group_median_map(train_frame, origin_col)
    dest_map = build_group_median_map(train_frame, dest_col)

    prior_values: List[np.ndarray] = []
    has_origin_values: List[bool] = []
    has_dest_values: List[bool] = []

    for _, row in result.iterrows():
        origin_key = str(row[origin_col])
        dest_key = str(row[dest_col])
        origin_vec = origin_map.get(origin_key)
        dest_vec = dest_map.get(dest_key)
        prior_vec, has_origin, has_dest = blend_origin_destination_prior(origin_vec, dest_vec, global_vec)
        prior_values.append(prior_vec)
        has_origin_values.append(has_origin)
        has_dest_values.append(has_dest)

    priors = np.vstack(prior_values).astype(np.float32)
    result["cold_prior_truck_q25"] = priors[:, 0]
    result["cold_prior_truck_q50"] = priors[:, 1]
    result["cold_prior_truck_q75"] = priors[:, 2]
    result["cold_prior_iqr"] = result["cold_prior_truck_q75"] - result["cold_prior_truck_q25"]
    result["cold_prior_q75_q50_gap"] = result["cold_prior_truck_q75"] - result["cold_prior_truck_q50"]
    result["cold_prior_q50_q25_gap"] = result["cold_prior_truck_q50"] - result["cold_prior_truck_q25"]
    result["cold_has_origin_prior"] = np.asarray(has_origin_values, dtype=np.float32)
    result["cold_has_dest_prior"] = np.asarray(has_dest_values, dtype=np.float32)
    result["cold_has_any_zone_prior"] = ((result["cold_has_origin_prior"] > 0) | (result["cold_has_dest_prior"] > 0)).astype(np.float32)
    return result


df = build_cold_priors(df)
print("Cold prior feature means:")
print(df[COLD_PRIOR_COLUMNS].mean(numeric_only=True))

Cold prior feature means:
cold_prior_truck_q25       1555.832642
cold_prior_truck_q50       2312.527588
cold_prior_truck_q75       3708.205811
cold_prior_iqr             2152.372803
cold_prior_q75_q50_gap     1395.678223
cold_prior_q50_q25_gap      756.694885
cold_has_origin_prior         1.000000
cold_has_dest_prior           1.000000
cold_has_any_zone_prior       1.000000
dtype: float32


## 9. Node features from training rows only

FAF-zone node features are built exclusively from training rows. This prevents validation/test labels from leaking into node representations.

In [9]:
NODE_AGG_FEATURES = [
    "log1p_tons_truck",
    "log1p_tons_rail",
    "log1p_tons_multimodal",
    "log1p_tmiles_truck",
    "log1p_tmiles_rail",
    "log1p_tmiles_multimodal",
    "tons_share_truck",
    "tons_share_rail",
    "tons_share_multimodal",
    "has_truck_demand",
    "has_rail_demand",
    "has_multimodal_demand",
    "n_modes_with_positive_tons",
]
NODE_AGG_FEATURES = [column for column in NODE_AGG_FEATURES if column in df.columns]


def build_zone_index(frame: pd.DataFrame) -> Dict[str, int]:
    """Build a stable FAF-zone-to-index mapping from all observed origins and destinations."""
    zones = sorted(set(frame[origin_col].astype(str)) | set(frame[dest_col].astype(str)))
    return {zone: idx for idx, zone in enumerate(zones)}


def build_node_features(frame: pd.DataFrame, zone_index: Mapping[str, int]) -> Tuple[pd.DataFrame, List[str]]:
    """Create train-only node features for FAF zones."""
    train_frame = frame[frame["cold_split"].eq("train")].copy()
    rows: List[Dict[str, Any]] = []
    for zone, idx in zone_index.items():
        origin_rows = train_frame[train_frame[origin_col].astype(str).eq(zone)]
        dest_rows = train_frame[train_frame[dest_col].astype(str).eq(zone)]
        row: Dict[str, Any] = {"faf_zone": zone, "node_idx": idx}
        row["zone_numeric"] = float(int(zone)) if str(zone).isdigit() else float(idx)
        row["out_train_rows"] = float(len(origin_rows))
        row["in_train_rows"] = float(len(dest_rows))
        row["total_train_rows"] = float(len(origin_rows) + len(dest_rows))
        row["log1p_out_train_rows"] = float(np.log1p(len(origin_rows)))
        row["log1p_in_train_rows"] = float(np.log1p(len(dest_rows)))
        row["log1p_total_train_rows"] = float(np.log1p(len(origin_rows) + len(dest_rows)))
        for feature in NODE_AGG_FEATURES:
            row[f"orig_mean_{feature}"] = float(origin_rows[feature].mean()) if len(origin_rows) else 0.0
            row[f"dest_mean_{feature}"] = float(dest_rows[feature].mean()) if len(dest_rows) else 0.0
        rows.append(row)
    node_df = pd.DataFrame(rows).sort_values("node_idx").reset_index(drop=True)
    node_feature_cols = [column for column in node_df.columns if column not in {"faf_zone", "node_idx"}]
    return node_df, node_feature_cols


def is_zone_code_or_degree_feature(column: str) -> bool:
    """Identify node features that may act as zone-code or train-degree shortcuts."""
    text = column.lower()
    if text == "zone_numeric":
        return True
    if "train_rows" in text:
        return True
    if "degree" in text:
        return True
    return False


def standardize_node_matrix(values: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Standardize a node-feature matrix across FAF nodes."""
    mean = values.mean(axis=0)
    std = values.std(axis=0)
    std = np.where(std < 1e-6, 1.0, std)
    return ((values - mean) / std).astype(np.float32), mean.astype(np.float32), std.astype(np.float32)


zone_to_idx = build_zone_index(df)
node_feature_df, node_feature_columns = build_node_features(df, zone_to_idx)

# Standard train-only node features.
standard_node_raw = node_feature_df[node_feature_columns].astype(float).to_numpy(dtype=np.float32)
standard_node_np, standard_node_mean, standard_node_std = standardize_node_matrix(standard_node_raw)

# Remove zone numeric and train-degree/count features to test shortcuts.
no_zone_degree_columns = [column for column in node_feature_columns if not is_zone_code_or_degree_feature(column)]
if not no_zone_degree_columns:
    raise ValueError("No node features remain after removing zone-code and degree features.")
no_zone_degree_raw = node_feature_df[no_zone_degree_columns].astype(float).to_numpy(dtype=np.float32)
no_zone_degree_np, no_zone_degree_mean, no_zone_degree_std = standardize_node_matrix(no_zone_degree_raw)

# Constant node features test whether graph topology plus edge features can help without node attributes.
constant_node_np = np.ones((len(zone_to_idx), 1), dtype=np.float32)

node_feature_variants_np: Dict[str, np.ndarray] = {
    "standard": standard_node_np,
    "no_zone_numeric_no_degree": no_zone_degree_np,
    "constant": constant_node_np,
}
node_feature_columns_by_variant: Dict[str, List[str]] = {
    "standard": list(node_feature_columns),
    "no_zone_numeric_no_degree": list(no_zone_degree_columns),
    "constant": ["constant_one"],
}
node_normalization_metadata = {
    "standard": {
        "mean": standard_node_mean.tolist(),
        "std": standard_node_std.tolist(),
    },
    "no_zone_numeric_no_degree": {
        "mean": no_zone_degree_mean.tolist(),
        "std": no_zone_degree_std.tolist(),
        "removed_columns": [column for column in node_feature_columns if is_zone_code_or_degree_feature(column)],
    },
    "constant": {
        "mean": [1.0],
        "std": [1.0],
    },
}

print("Number of FAF nodes:", len(zone_to_idx))
print("Standard node feature dimension:", len(node_feature_columns))
print("No-zone/no-degree node feature dimension:", len(no_zone_degree_columns))
print("Removed shortcut-like node features:", node_normalization_metadata["no_zone_numeric_no_degree"]["removed_columns"])
print(node_feature_df.head())


Number of FAF nodes: 104
Standard node feature dimension: 33
No-zone/no-degree node feature dimension: 26
Removed shortcut-like node features: ['zone_numeric', 'out_train_rows', 'in_train_rows', 'total_train_rows', 'log1p_out_train_rows', 'log1p_in_train_rows', 'log1p_total_train_rows']
  faf_zone  node_idx  zone_numeric  out_train_rows  in_train_rows  \
0      011         0          11.0           413.0          391.0   
1      012         1          12.0           392.0          410.0   
2      019         2          19.0           405.0          410.0   
3      050         3          50.0           408.0          436.0   
4      091         4          91.0           388.0          454.0   

   total_train_rows  log1p_out_train_rows  log1p_in_train_rows  \
0             804.0              6.025866             5.971262   
1             802.0              5.973810             6.018593   
2             815.0              6.006353             6.018593   
3             844.0              

## 10. Numeric preprocessing for supervised edge features

The edge feature matrix uses the 64 manifest features plus Cold-OD fallback prior features. Median imputation and z-score scaling are fitted only on training rows.

In [10]:
@dataclass
class NumericPreprocessor:
    """Train-only numeric median imputer and z-score scaler."""
    columns: List[str]
    medians: Dict[str, float]
    means: Dict[str, float]
    stds: Dict[str, float]

    @classmethod
    def fit(cls, frame: pd.DataFrame, columns: Sequence[str]) -> "NumericPreprocessor":
        train_part = frame[frame["cold_split"].eq("train")]
        medians: Dict[str, float] = {}
        means: Dict[str, float] = {}
        stds: Dict[str, float] = {}
        for column in columns:
            values = pd.to_numeric(train_part[column], errors="coerce")
            median = float(values.median()) if values.notna().any() else 0.0
            filled = values.fillna(median).astype(float)
            mean = float(filled.mean())
            std = float(filled.std(ddof=0))
            if not np.isfinite(std) or std < 1e-8:
                std = 1.0
            medians[column] = median
            means[column] = mean
            stds[column] = std
        return cls(list(columns), medians, means, stds)

    def transform(self, frame: pd.DataFrame) -> np.ndarray:
        arrays: List[np.ndarray] = []
        for column in self.columns:
            values = pd.to_numeric(frame[column], errors="coerce").fillna(self.medians[column]).astype(float)
            scaled = (values - self.means[column]) / self.stds[column]
            arrays.append(scaled.to_numpy(dtype=np.float32))
        return np.vstack(arrays).T.astype(np.float32)

    def to_dict(self) -> Dict[str, Any]:
        return {
            "columns": self.columns,
            "medians": self.medians,
            "means": self.means,
            "stds": self.stds,
        }


edge_feature_columns = list(manifest_feature_columns) + list(COLD_PRIOR_COLUMNS)
edge_preprocessor = NumericPreprocessor.fit(df, edge_feature_columns)
edge_features_np = edge_preprocessor.transform(df)
print("Edge feature dimension:", edge_features_np.shape[1])

Edge feature dimension: 73


## 11. Build graph edges and ablation graph variants

This cell creates the full FAF-zone graph from training rows only, then derives all requested ablation graphs. Validation/test supervised OD edges are **not** used as message-passing edges.

In [11]:
def make_edge_index_from_pairs(pairs: Iterable[Tuple[str, str]], zone_index: Mapping[str, int]) -> torch.Tensor:
    """Build a PyTorch edge_index tensor from zone-code pairs."""
    src: List[int] = []
    dst: List[int] = []
    for orig, dest in pairs:
        if orig in zone_index and dest in zone_index:
            src.append(zone_index[orig])
            dst.append(zone_index[dest])
    if not src:
        return torch.empty((2, 0), dtype=torch.long)
    return torch.tensor([src, dst], dtype=torch.long)


def unique_pairs_from_mask(frame: pd.DataFrame, mask: np.ndarray, origin_col_name: str, dest_col_name: str) -> List[Tuple[str, str]]:
    """Return sorted unique origin-destination pairs selected by a boolean mask."""
    part = frame.loc[mask, [origin_col_name, dest_col_name]].drop_duplicates()
    return list(map(tuple, part.astype(str).to_numpy()))


def add_forward_reverse_relation(
    edge_dict: Dict[Tuple[str, str, str], torch.Tensor],
    relation_name: str,
    pairs: Iterable[Tuple[str, str]],
    zone_index: Mapping[str, int],
) -> None:
    """Add a forward relation and its reverse relation to an edge dictionary."""
    forward = make_edge_index_from_pairs(pairs, zone_index)
    reverse = torch.stack([forward[1], forward[0]], dim=0) if forward.numel() > 0 else forward.clone()
    edge_dict[("faf", relation_name, "faf")] = forward
    edge_dict[("faf", f"rev_{relation_name}", "faf")] = reverse


def add_self_loop_relation(edge_dict: Dict[Tuple[str, str, str], torch.Tensor], n_nodes: int) -> None:
    """Add one self-loop edge per FAF node."""
    self_edges = torch.arange(n_nodes, dtype=torch.long).unsqueeze(0).repeat(2, 1)
    edge_dict[("faf", "self_loop", "faf")] = self_edges


def build_full_typed_edges(frame: pd.DataFrame, zone_index: Mapping[str, int]) -> Dict[Tuple[str, str, str], torch.Tensor]:
    """Build the full FAF-zone typed graph from training rows only."""
    train_part = frame.loc[frame["cold_split"].eq("train")].copy()
    edge_dict: Dict[Tuple[str, str, str], torch.Tensor] = {}

    train_pairs = unique_pairs_from_mask(train_part, np.ones(len(train_part), dtype=bool), origin_col, dest_col)
    add_forward_reverse_relation(edge_dict, "train_od", train_pairs, zone_index)

    relation_columns = {
        "demand_truck": "has_truck_demand",
        "demand_rail": "has_rail_demand",
        "demand_multimodal": "has_multimodal_demand",
    }
    for relation_name, flag_col in relation_columns.items():
        if flag_col in train_part.columns:
            flags = pd.to_numeric(train_part[flag_col], errors="coerce").fillna(0.0).to_numpy()
            pairs = unique_pairs_from_mask(train_part, flags > 0, origin_col, dest_col)
            add_forward_reverse_relation(edge_dict, relation_name, pairs, zone_index)

    add_self_loop_relation(edge_dict, len(zone_index))
    return edge_dict


def subset_edge_dict(edge_dict: Mapping[Tuple[str, str, str], torch.Tensor], keep_relations: Sequence[str]) -> Dict[Tuple[str, str, str], torch.Tensor]:
    """Keep only selected relation names from a typed edge dictionary."""
    keep_set = set(keep_relations)
    return {edge_type: edge_index.clone() for edge_type, edge_index in edge_dict.items() if edge_type[1] in keep_set}


def build_homogeneous_edge_index(edge_dict: Mapping[Tuple[str, str, str], torch.Tensor]) -> torch.Tensor:
    """Merge typed edges into a unique homogeneous edge index for GraphSAGE."""
    edge_indices = [edge_index for edge_index in edge_dict.values() if edge_index.numel() > 0]
    if not edge_indices:
        return torch.empty((2, 0), dtype=torch.long)
    merged = torch.cat(edge_indices, dim=1)
    return torch.unique(merged, dim=1)


def shuffled_edge_dict(edge_dict: Mapping[Tuple[str, str, str], torch.Tensor], n_nodes: int, seed: int) -> Dict[Tuple[str, str, str], torch.Tensor]:
    """Randomize edge endpoints within each non-self relation while preserving relation counts."""
    rng = np.random.default_rng(seed)
    result: Dict[Tuple[str, str, str], torch.Tensor] = {}
    for edge_type, edge_index in edge_dict.items():
        n_edges = edge_index.shape[1]
        if edge_type[1] == "self_loop" or n_edges == 0:
            result[edge_type] = edge_index.clone()
            continue
        src = rng.integers(0, n_nodes, size=n_edges, endpoint=False, dtype=np.int64)
        dst = rng.integers(0, n_nodes, size=n_edges, endpoint=False, dtype=np.int64)
        result[edge_type] = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
    return result


def randomized_edge_types(edge_dict: Mapping[Tuple[str, str, str], torch.Tensor], seed: int) -> Dict[Tuple[str, str, str], torch.Tensor]:
    """Keep endpoints but randomly reassign non-self edges among existing relation types."""
    rng = np.random.default_rng(seed)
    self_items = {key: value.clone() for key, value in edge_dict.items() if key[1] == "self_loop"}
    non_self_keys = [key for key in edge_dict.keys() if key[1] != "self_loop"]
    if not non_self_keys:
        return dict(self_items)

    all_edges: List[torch.Tensor] = []
    counts: List[int] = []
    for key in non_self_keys:
        edge_index = edge_dict[key]
        counts.append(edge_index.shape[1])
        if edge_index.numel() > 0:
            all_edges.append(edge_index)
    if not all_edges:
        return {key: edge_dict[key].clone() for key in edge_dict}

    merged = torch.cat(all_edges, dim=1)
    perm = torch.tensor(rng.permutation(merged.shape[1]), dtype=torch.long)
    merged = merged[:, perm]

    result: Dict[Tuple[str, str, str], torch.Tensor] = {}
    offset = 0
    for key, count in zip(non_self_keys, counts):
        result[key] = merged[:, offset:offset + count].clone()
        offset += count
    result.update(self_items)
    return result


FULL_RELATIONS = [
    "train_od", "rev_train_od",
    "demand_truck", "rev_demand_truck",
    "demand_rail", "rev_demand_rail",
    "demand_multimodal", "rev_demand_multimodal",
    "self_loop",
]
TRAIN_OD_ONLY_RELATIONS = ["train_od", "rev_train_od", "self_loop"]
DEMAND_ONLY_RELATIONS = [
    "demand_truck", "rev_demand_truck",
    "demand_rail", "rev_demand_rail",
    "demand_multimodal", "rev_demand_multimodal",
    "self_loop",
]
FULL_MINUS_TRAIN_OD_RELATIONS = [relation for relation in FULL_RELATIONS if relation not in {"train_od", "rev_train_od"}]
TRUCK_ONLY_RELATIONS = ["demand_truck", "rev_demand_truck", "self_loop"]
RAIL_ONLY_RELATIONS = ["demand_rail", "rev_demand_rail", "self_loop"]
MULTIMODAL_ONLY_RELATIONS = ["demand_multimodal", "rev_demand_multimodal", "self_loop"]
TRUCK_PLUS_RAIL_RELATIONS = ["demand_truck", "rev_demand_truck", "demand_rail", "rev_demand_rail", "self_loop"]
TRUCK_PLUS_MULTIMODAL_RELATIONS = ["demand_truck", "rev_demand_truck", "demand_multimodal", "rev_demand_multimodal", "self_loop"]
RAIL_PLUS_MULTIMODAL_RELATIONS = ["demand_rail", "rev_demand_rail", "demand_multimodal", "rev_demand_multimodal", "self_loop"]
SELF_LOOP_ONLY_RELATIONS = ["self_loop"]

full_typed_edge_index_dict_cpu = build_full_typed_edges(df, zone_to_idx)

ablation_edge_dicts_cpu: Dict[str, Dict[Tuple[str, str, str], torch.Tensor]] = {
    "hgt_full_faf_zone_graph": subset_edge_dict(full_typed_edge_index_dict_cpu, FULL_RELATIONS),
    "hgt_train_od_only": subset_edge_dict(full_typed_edge_index_dict_cpu, TRAIN_OD_ONLY_RELATIONS),
    "hgt_demand_edges_only": subset_edge_dict(full_typed_edge_index_dict_cpu, DEMAND_ONLY_RELATIONS),
    "hgt_self_loop_only": subset_edge_dict(full_typed_edge_index_dict_cpu, SELF_LOOP_ONLY_RELATIONS),
    "hgt_no_zone_numeric_no_degree_features": subset_edge_dict(full_typed_edge_index_dict_cpu, FULL_RELATIONS),
    "hgt_constant_node_features_full_graph": subset_edge_dict(full_typed_edge_index_dict_cpu, FULL_RELATIONS),
    "hgt_train_od_edges_removed_but_demand_kept": subset_edge_dict(full_typed_edge_index_dict_cpu, FULL_MINUS_TRAIN_OD_RELATIONS),
    "hgt_truck_demand_only": subset_edge_dict(full_typed_edge_index_dict_cpu, TRUCK_ONLY_RELATIONS),
    "hgt_rail_demand_only": subset_edge_dict(full_typed_edge_index_dict_cpu, RAIL_ONLY_RELATIONS),
    "hgt_multimodal_demand_only": subset_edge_dict(full_typed_edge_index_dict_cpu, MULTIMODAL_ONLY_RELATIONS),
    "hgt_truck_plus_rail_demand": subset_edge_dict(full_typed_edge_index_dict_cpu, TRUCK_PLUS_RAIL_RELATIONS),
    "hgt_truck_plus_multimodal_demand": subset_edge_dict(full_typed_edge_index_dict_cpu, TRUCK_PLUS_MULTIMODAL_RELATIONS),
    "hgt_rail_plus_multimodal_demand": subset_edge_dict(full_typed_edge_index_dict_cpu, RAIL_PLUS_MULTIMODAL_RELATIONS),
}

for replica_idx, perturb_seed in enumerate(cfg.graph_shuffle_seeds, start=1):
    ablation_edge_dicts_cpu[f"hgt_shuffled_graph_edges_s{replica_idx}"] = shuffled_edge_dict(
        full_typed_edge_index_dict_cpu, len(zone_to_idx), perturb_seed
    )

for replica_idx, perturb_seed in enumerate(cfg.edge_type_randomization_seeds, start=1):
    ablation_edge_dicts_cpu[f"hgt_randomized_edge_types_s{replica_idx}"] = randomized_edge_types(
        full_typed_edge_index_dict_cpu, perturb_seed
    )

hom_full_edge_index_cpu = build_homogeneous_edge_index(full_typed_edge_index_dict_cpu)

edge_count_rows: List[Dict[str, Any]] = []
for variant_name, edge_dict in ablation_edge_dicts_cpu.items():
    for edge_type, edge_index in edge_dict.items():
        edge_count_rows.append({
            "variant": variant_name,
            "edge_type": "::".join(edge_type),
            "n_edges": int(edge_index.shape[1]),
        })
edge_count_table = pd.DataFrame(edge_count_rows)

print("Full typed edge counts:")
for edge_type, edge_index in full_typed_edge_index_dict_cpu.items():
    print(f"  {edge_type}: {edge_index.shape[1]}")
print("Homogeneous full edge count:", hom_full_edge_index_cpu.shape[1])
print("Ablation edge variants:", list(ablation_edge_dicts_cpu))


Full typed edge counts:
  ('faf', 'train_od', 'faf'): 8748
  ('faf', 'rev_train_od', 'faf'): 8748
  ('faf', 'demand_truck', 'faf'): 8693
  ('faf', 'rev_demand_truck', 'faf'): 8693
  ('faf', 'demand_rail', 'faf'): 5834
  ('faf', 'rev_demand_rail', 'faf'): 5834
  ('faf', 'demand_multimodal', 'faf'): 8692
  ('faf', 'rev_demand_multimodal', 'faf'): 8692
  ('faf', 'self_loop', 'faf'): 104
Homogeneous full edge count: 10412
Ablation edge variants: ['hgt_full_faf_zone_graph', 'hgt_train_od_only', 'hgt_demand_edges_only', 'hgt_self_loop_only', 'hgt_no_zone_numeric_no_degree_features', 'hgt_constant_node_features_full_graph', 'hgt_train_od_edges_removed_but_demand_kept', 'hgt_truck_demand_only', 'hgt_rail_demand_only', 'hgt_multimodal_demand_only', 'hgt_truck_plus_rail_demand', 'hgt_truck_plus_multimodal_demand', 'hgt_rail_plus_multimodal_demand', 'hgt_shuffled_graph_edges_s1', 'hgt_shuffled_graph_edges_s2', 'hgt_shuffled_graph_edges_s3', 'hgt_shuffled_graph_edges_s4', 'hgt_shuffled_graph_edges

## 12. Build supervised edge tensors

The supervised unit is still an OD-year edge. Graph message-passing edges are train-only graph edges, while supervised prediction edges are the train/validation/test OD-year rows.

In [12]:
def build_year_index(frame: pd.DataFrame, year_col_name: str) -> Tuple[np.ndarray, Dict[int, int]]:
    """Map years to compact embedding indices."""
    years = sorted(frame[year_col_name].astype(int).unique().tolist())
    mapping = {year: idx for idx, year in enumerate(years)}
    return frame[year_col_name].astype(int).map(mapping).to_numpy(dtype=np.int64), mapping


def build_edge_label_index(frame: pd.DataFrame, zone_index: Mapping[str, int]) -> np.ndarray:
    """Build edge_label_index for supervised OD-year rows."""
    src = frame[origin_col].astype(str).map(zone_index).to_numpy(dtype=np.int64)
    dst = frame[dest_col].astype(str).map(zone_index).to_numpy(dtype=np.int64)
    return np.vstack([src, dst]).astype(np.int64)


year_index_np, year_to_idx = build_year_index(df, year_col)
edge_label_index_np = build_edge_label_index(df, zone_to_idx)
y_np = df[LABEL_COLUMNS].to_numpy(dtype=np.float32)
base_prior_np = df[["cold_prior_truck_q25", "cold_prior_truck_q50", "cold_prior_truck_q75"]].to_numpy(dtype=np.float32)

train_q50_median = float(np.median(df.loc[df["cold_split"].eq("train"), "truck_q50"]))
target_scale = train_q50_median if train_q50_median > 0 else 1.0

split_indices_np = {
    split_name: df.index[df["cold_split"].eq(split_name)].to_numpy(dtype=np.int64)
    for split_name in ["train", "val", "test"]
}

print("Target scale:", target_scale)
print("Split sizes:", {key: len(value) for key, value in split_indices_np.items()})

Target scale: 2317.03
Split sizes: {'train': 42849, 'val': 957, 'test': 1057}


## 13. Move tensors to device

This cell prepares CPU and device tensors. Graph edge dictionaries are moved to the selected device separately for each ablation variant.

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

node_x_variants_cpu: Dict[str, torch.Tensor] = {
    name: torch.tensor(values, dtype=torch.float32)
    for name, values in node_feature_variants_np.items()
}
node_x_variants: Dict[str, torch.Tensor] = {
    name: tensor.to(device)
    for name, tensor in node_x_variants_cpu.items()
}

# Backward-compatible alias for the standard variant.
node_x_cpu = node_x_variants_cpu["standard"]
node_x = node_x_variants["standard"]

edge_attr_cpu = torch.tensor(edge_features_np, dtype=torch.float32)
edge_label_index_cpu = torch.tensor(edge_label_index_np, dtype=torch.long)
y_cpu = torch.tensor(y_np / target_scale, dtype=torch.float32)
base_prior_scaled_cpu = torch.tensor(base_prior_np / target_scale, dtype=torch.float32)
year_idx_cpu = torch.tensor(year_index_np, dtype=torch.long)
weights_cpu = torch.tensor(df["sample_weight"].to_numpy(dtype=np.float32), dtype=torch.float32)

edge_attr = edge_attr_cpu.to(device)
edge_label_index = edge_label_index_cpu.to(device)
y_scaled = y_cpu.to(device)
base_prior_scaled = base_prior_scaled_cpu.to(device)
year_idx = year_idx_cpu.to(device)
weights = weights_cpu.to(device)

split_indices = {key: torch.tensor(value, dtype=torch.long, device=device) for key, value in split_indices_np.items()}

print("Node feature variants:", {name: tuple(tensor.shape) for name, tensor in node_x_variants.items()})
print("Edge feature tensor:", tuple(edge_attr.shape))
print("Supervised edge_label_index:", tuple(edge_label_index.shape))


Using device: cuda
Node feature variants: {'standard': (104, 33), 'no_zone_numeric_no_degree': (104, 26), 'constant': (104, 1)}
Edge feature tensor: (73972, 73)
Supervised edge_label_index: (2, 73972)


C:\Users\Nick_James\AppData\Local\Temp\ipykernel_16436\1417006522.py:9: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  name: tensor.to(device)


## 14. Loss and metric utilities

The training objective is weighted pinball loss plus an optional IQR auxiliary loss. Metrics are reported in original minutes.

In [14]:
def inv_softplus(x: torch.Tensor) -> torch.Tensor:
    """Numerically stable inverse softplus for positive x."""
    x = torch.clamp(x, min=1e-8)
    return torch.where(x > 20.0, x, torch.log(torch.expm1(x)))


def pinball_loss_torch(pred: torch.Tensor, target: torch.Tensor, taus: torch.Tensor) -> torch.Tensor:
    """Return elementwise pinball loss for scaled predictions."""
    taus = taus.to(pred.device).view(1, -1)
    diff = target - pred
    return torch.maximum(taus * diff, (taus - 1.0) * diff)


def training_loss(pred_scaled: torch.Tensor, target_scaled: torch.Tensor, row_weights: torch.Tensor, lambda_iqr: float) -> torch.Tensor:
    """Weighted quantile loss plus optional IQR SmoothL1 loss."""
    q_loss = pinball_loss_torch(pred_scaled, target_scaled, TAUS_TENSOR).mean(dim=1)
    weighted_q = (q_loss * row_weights).mean()
    pred_iqr = pred_scaled[:, 2] - pred_scaled[:, 0]
    true_iqr = target_scaled[:, 2] - target_scaled[:, 0]
    iqr_loss = F.smooth_l1_loss(pred_iqr, true_iqr)
    return weighted_q + lambda_iqr * iqr_loss


def pinball_numpy(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """Return row-by-quantile pinball losses in minutes."""
    diff = y_true - y_pred
    taus = TAU_VALUES.reshape(1, -1)
    return np.maximum(taus * diff, (taus - 1.0) * diff)


def compute_metrics_from_arrays(y_true: np.ndarray, y_pred: np.ndarray, row_weights: Optional[np.ndarray] = None) -> Dict[str, float]:
    """Compute standard FREIGHT-MNet quantile metrics."""
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    if y_true.shape[0] == 0:
        return {"n_rows": 0}

    abs_error = np.abs(y_pred - y_true)
    pin = pinball_numpy(y_true, y_pred)
    pin_row = pin.mean(axis=1)
    true_iqr = y_true[:, 2] - y_true[:, 0]
    pred_iqr = y_pred[:, 2] - y_pred[:, 0]
    iqr_abs = np.abs(pred_iqr - true_iqr)

    if row_weights is None:
        row_weights = np.ones(y_true.shape[0], dtype=np.float32)
    row_weights = np.asarray(row_weights, dtype=np.float32)
    if row_weights.sum() <= 0:
        row_weights = np.ones_like(row_weights)

    def wavg(values: np.ndarray) -> float:
        return float(np.average(values, weights=row_weights))

    metrics = {
        "n_rows": int(y_true.shape[0]),
        "mae_q25": float(abs_error[:, 0].mean()),
        "mae_q50": float(abs_error[:, 1].mean()),
        "mae_q75": float(abs_error[:, 2].mean()),
        "rmse_q50": float(np.sqrt(np.mean((y_pred[:, 1] - y_true[:, 1]) ** 2))),
        "pinball_q25": float(pin[:, 0].mean()),
        "pinball_q50": float(pin[:, 1].mean()),
        "pinball_q75": float(pin[:, 2].mean()),
        "pinball_mean": float(pin_row.mean()),
        "weighted_pinball_mean": wavg(pin_row),
        "iqr_mae": float(iqr_abs.mean()),
        "bias_q25": float((y_pred[:, 0] - y_true[:, 0]).mean()),
        "bias_q50": float((y_pred[:, 1] - y_true[:, 1]).mean()),
        "bias_q75": float((y_pred[:, 2] - y_true[:, 2]).mean()),
        "pred_iqr_mean": float(pred_iqr.mean()),
        "true_iqr_mean": float(true_iqr.mean()),
        "raw_crossing_rate": float(np.mean((y_pred[:, 0] > y_pred[:, 1]) | (y_pred[:, 1] > y_pred[:, 2]))),
        "q50_inside_pred_q25_q75_rate": float(np.mean((y_true[:, 1] >= y_pred[:, 0]) & (y_true[:, 1] <= y_pred[:, 2]))),
    }

    q75_threshold = float(np.quantile(y_true[:, 2], 0.90))
    stress_mask = y_true[:, 2] >= q75_threshold
    metrics["stress_top10_threshold_q75"] = q75_threshold
    metrics["stress_top10_n_rows_q75"] = int(stress_mask.sum())
    metrics["stress_top10_mae_q75"] = float(abs_error[stress_mask, 2].mean()) if stress_mask.any() else np.nan
    metrics["stress_top10_iqr_mae"] = float(iqr_abs[stress_mask].mean()) if stress_mask.any() else np.nan

    iqr_threshold = float(np.quantile(true_iqr, 0.90))
    top_iqr_mask = true_iqr >= iqr_threshold
    metrics["top_iqr10_threshold"] = iqr_threshold
    metrics["top_iqr10_n_rows"] = int(top_iqr_mask.sum())
    metrics["top_iqr10_mae_q75"] = float(abs_error[top_iqr_mask, 2].mean()) if top_iqr_mask.any() else np.nan
    metrics["top_iqr10_iqr_mae"] = float(iqr_abs[top_iqr_mask].mean()) if top_iqr_mask.any() else np.nan
    return metrics


def compute_metrics_for_indices(pred_minutes: np.ndarray, indices_np: np.ndarray) -> Dict[str, float]:
    """Compute metrics for selected original row indices."""
    true_minutes = y_np[indices_np]
    row_weights = df.iloc[indices_np]["sample_weight"].to_numpy(dtype=np.float32)
    return compute_metrics_from_arrays(true_minutes, pred_minutes, row_weights=row_weights)

## 15. Model definitions

The graph and MLP baselines share the same residualized monotone quantile head. This keeps the ablation focused on the graph structure rather than on differences in output parameterization.

In [15]:
class ResidualizedMonotoneHead(nn.Module):
    """Residualized monotone quantile head around a base prior."""

    def __init__(self, hidden_dim: int):
        super().__init__()
        self.out = nn.Linear(hidden_dim, 3)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, hidden: torch.Tensor, base_scaled: torch.Tensor) -> torch.Tensor:
        """Return scaled q25/q50/q75 predictions with guaranteed monotonicity."""
        delta = self.out(hidden)
        base_q25 = torch.clamp(base_scaled[:, 0], min=1e-5)
        base_gap50 = torch.clamp(base_scaled[:, 1] - base_scaled[:, 0], min=1e-5)
        base_gap75 = torch.clamp(base_scaled[:, 2] - base_scaled[:, 1], min=1e-5)
        q25 = F.softplus(inv_softplus(base_q25) + delta[:, 0])
        gap50 = F.softplus(inv_softplus(base_gap50) + delta[:, 1])
        gap75 = F.softplus(inv_softplus(base_gap75) + delta[:, 2])
        q50 = q25 + gap50
        q75 = q50 + gap75
        return torch.stack([q25, q50, q75], dim=-1)


class EdgeDecoder(nn.Module):
    """Decode source/destination embeddings and OD-year features into residual quantiles."""

    def __init__(self, hidden_dim: int, edge_dim: int, year_count: int, year_embedding_dim: int, edge_hidden_dim: int, dropout: float):
        super().__init__()
        self.year_embedding = nn.Embedding(year_count, year_embedding_dim)
        input_dim = hidden_dim * 4 + edge_dim + year_embedding_dim
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, edge_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(edge_hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.head = ResidualizedMonotoneHead(hidden_dim)

    def forward(self, src_h: torch.Tensor, dst_h: torch.Tensor, edge_attr_batch: torch.Tensor, base_scaled: torch.Tensor, year_idx_batch: torch.Tensor) -> torch.Tensor:
        year_h = self.year_embedding(year_idx_batch)
        z = torch.cat([src_h, dst_h, torch.abs(src_h - dst_h), src_h * dst_h, edge_attr_batch, year_h], dim=-1)
        hidden = self.mlp(z)
        return self.head(hidden, base_scaled)


class EdgeDecoderNoEdgeFeatures(nn.Module):
    """Decode source/destination embeddings and year only, without OD edge features."""

    def __init__(self, hidden_dim: int, year_count: int, year_embedding_dim: int, edge_hidden_dim: int, dropout: float):
        super().__init__()
        self.year_embedding = nn.Embedding(year_count, year_embedding_dim)
        input_dim = hidden_dim * 4 + year_embedding_dim
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, edge_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(edge_hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.head = ResidualizedMonotoneHead(hidden_dim)

    def forward(self, src_h: torch.Tensor, dst_h: torch.Tensor, base_scaled: torch.Tensor, year_idx_batch: torch.Tensor) -> torch.Tensor:
        year_h = self.year_embedding(year_idx_batch)
        z = torch.cat([src_h, dst_h, torch.abs(src_h - dst_h), src_h * dst_h, year_h], dim=-1)
        hidden = self.mlp(z)
        return self.head(hidden, base_scaled)


class GraphSAGEResidualQuantileModel(nn.Module):
    """Homogeneous GraphSAGE residual OD quantile predictor."""

    def __init__(self, node_in_dim: int, edge_dim: int, year_count: int, config: ExperimentConfig):
        super().__init__()
        self.node_proj = nn.Linear(node_in_dim, config.hidden_dim)
        self.convs = nn.ModuleList([SAGEConv(config.hidden_dim, config.hidden_dim) for _ in range(config.graph_layers)])
        self.dropout = nn.Dropout(config.dropout)
        self.decoder = EdgeDecoder(config.hidden_dim, edge_dim, year_count, config.year_embedding_dim, config.edge_hidden_dim, config.dropout)

    def encode(self, node_features: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = F.gelu(self.node_proj(node_features))
        for conv in self.convs:
            h = self.dropout(F.gelu(conv(h, edge_index)))
        return h

    def forward(self, node_features: torch.Tensor, edge_index: torch.Tensor, edge_label_index_batch: torch.Tensor, edge_attr_batch: torch.Tensor, base_scaled_batch: torch.Tensor, year_idx_batch: torch.Tensor) -> torch.Tensor:
        h = self.encode(node_features, edge_index)
        src, dst = edge_label_index_batch
        return self.decoder(h[src], h[dst], edge_attr_batch, base_scaled_batch, year_idx_batch)


class HGTResidualQuantileModel(nn.Module):
    """Typed-relation HGT residual OD quantile predictor."""

    def __init__(self, node_in_dim: int, edge_dim: int, year_count: int, metadata: Tuple[List[str], List[Tuple[str, str, str]]], config: ExperimentConfig):
        super().__init__()
        self.node_proj = nn.ModuleDict({"faf": nn.Linear(node_in_dim, config.hidden_dim)})
        self.convs = nn.ModuleList([
            HGTConv(config.hidden_dim, config.hidden_dim, metadata, heads=config.hgt_heads)
            for _ in range(config.graph_layers)
        ])
        self.dropout = nn.Dropout(config.dropout)
        self.decoder = EdgeDecoder(config.hidden_dim, edge_dim, year_count, config.year_embedding_dim, config.edge_hidden_dim, config.dropout)

    def encode(self, x_dict: Mapping[str, torch.Tensor], edge_index_dict: Mapping[Tuple[str, str, str], torch.Tensor]) -> Dict[str, torch.Tensor]:
        h_dict = {node_type: F.gelu(self.node_proj[node_type](x)) for node_type, x in x_dict.items()}
        for conv in self.convs:
            h_dict = conv(h_dict, edge_index_dict)
            h_dict = {node_type: self.dropout(F.gelu(h)) for node_type, h in h_dict.items()}
        return h_dict

    def forward(self, x_dict: Mapping[str, torch.Tensor], edge_index_dict: Mapping[Tuple[str, str, str], torch.Tensor], edge_label_index_batch: torch.Tensor, edge_attr_batch: torch.Tensor, base_scaled_batch: torch.Tensor, year_idx_batch: torch.Tensor) -> torch.Tensor:
        h_dict = self.encode(x_dict, edge_index_dict)
        src, dst = edge_label_index_batch
        h = h_dict["faf"]
        return self.decoder(h[src], h[dst], edge_attr_batch, base_scaled_batch, year_idx_batch)


class NodeFeatureEdgeDecoderResidualQuantileModel(nn.Module):
    """No-message-passing model using origin/destination node features plus edge features."""

    def __init__(self, node_in_dim: int, edge_dim: int, year_count: int, config: ExperimentConfig):
        super().__init__()
        self.node_proj = nn.Sequential(
            nn.Linear(node_in_dim, config.hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.hidden_dim),
            nn.GELU(),
        )
        self.decoder = EdgeDecoder(config.hidden_dim, edge_dim, year_count, config.year_embedding_dim, config.edge_hidden_dim, config.dropout)

    def forward(self, node_features: torch.Tensor, edge_label_index_batch: torch.Tensor, edge_attr_batch: torch.Tensor, base_scaled_batch: torch.Tensor, year_idx_batch: torch.Tensor) -> torch.Tensor:
        h = self.node_proj(node_features)
        src, dst = edge_label_index_batch
        return self.decoder(h[src], h[dst], edge_attr_batch, base_scaled_batch, year_idx_batch)


class NodeIdEmbeddingOnlyResidualQuantileModel(nn.Module):
    """No-graph model using only learned origin/destination FAF embeddings and year."""

    def __init__(self, n_nodes: int, year_count: int, config: ExperimentConfig):
        super().__init__()
        self.embedding = nn.Embedding(n_nodes, config.hidden_dim)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.02)
        self.decoder = EdgeDecoderNoEdgeFeatures(config.hidden_dim, year_count, config.year_embedding_dim, config.edge_hidden_dim, config.dropout)

    def forward(self, edge_label_index_batch: torch.Tensor, base_scaled_batch: torch.Tensor, year_idx_batch: torch.Tensor) -> torch.Tensor:
        src, dst = edge_label_index_batch
        h = self.embedding.weight
        return self.decoder(h[src], h[dst], base_scaled_batch, year_idx_batch)


class MLPResidualQuantileModel(nn.Module):
    """No-graph MLP residual OD quantile predictor using only OD edge features."""

    def __init__(self, edge_dim: int, year_count: int, config: ExperimentConfig):
        super().__init__()
        self.year_embedding = nn.Embedding(year_count, config.year_embedding_dim)
        input_dim = edge_dim + config.year_embedding_dim
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, config.edge_hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.edge_hidden_dim, config.hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
        )
        self.head = ResidualizedMonotoneHead(config.hidden_dim)

    def forward(self, edge_attr_batch: torch.Tensor, base_scaled_batch: torch.Tensor, year_idx_batch: torch.Tensor) -> torch.Tensor:
        year_h = self.year_embedding(year_idx_batch)
        hidden = self.mlp(torch.cat([edge_attr_batch, year_h], dim=-1))
        return self.head(hidden, base_scaled_batch)


## 16. Ablation specification objects

Each ablation specification defines a model type and the graph it is allowed to use. This makes the training loop generic and prevents special-case code from changing the loss, features, or checkpoint logic across ablations.

In [16]:
@dataclass
class AblationSpec:
    """Description and graph data for one ablation run."""
    name: str
    model_type: str  # hgt, graphsage, mlp, node_feature_decoder, or node_id_embedding
    description: str
    typed_edge_dict_cpu: Optional[Dict[Tuple[str, str, str], torch.Tensor]] = None
    hom_edge_index_cpu: Optional[torch.Tensor] = None
    node_feature_variant: str = "standard"
    perturbation_seed: Optional[int] = None

    def metadata(self) -> Optional[Tuple[List[str], List[Tuple[str, str, str]]]]:
        if self.typed_edge_dict_cpu is None:
            return None
        return (["faf"], list(self.typed_edge_dict_cpu.keys()))


def hgt_spec(name: str, description: str, edge_variant: Optional[str] = None, node_feature_variant: str = "standard", perturbation_seed: Optional[int] = None) -> AblationSpec:
    """Convenience constructor for HGT ablation specifications."""
    edge_variant = edge_variant or name
    return AblationSpec(
        name=name,
        model_type="hgt",
        description=description,
        typed_edge_dict_cpu=ablation_edge_dicts_cpu[edge_variant],
        node_feature_variant=node_feature_variant,
        perturbation_seed=perturbation_seed,
    )


def build_ablation_specs() -> Dict[str, AblationSpec]:
    """Create all graph, node-control, perturbation, and no-graph ablation specifications."""
    specs: Dict[str, AblationSpec] = {}

    # Main graph and baseline controls.
    specs["hgt_full_faf_zone_graph"] = hgt_spec(
        "hgt_full_faf_zone_graph",
        "HGT with train_od, all demand-mode edges, reverse edges, and self-loops.",
    )
    specs["graphsage_same_edges"] = AblationSpec(
        name="graphsage_same_edges",
        model_type="graphsage",
        description="Homogeneous GraphSAGE using the same full FAF-zone graph as the full HGT variant.",
        hom_edge_index_cpu=hom_full_edge_index_cpu,
        node_feature_variant="standard",
    )
    specs["mlp_same_features_no_graph"] = AblationSpec(
        name="mlp_same_features_no_graph",
        model_type="mlp",
        description="No-graph MLP residual model using OD edge features and fallback prior only.",
    )
    specs["node_features_only_edge_decoder"] = AblationSpec(
        name="node_features_only_edge_decoder",
        model_type="node_feature_decoder",
        description="No message passing. Uses origin node features, destination node features, OD edge features, and fallback prior.",
        node_feature_variant="standard",
    )
    specs["node_id_embedding_only"] = AblationSpec(
        name="node_id_embedding_only",
        model_type="node_id_embedding",
        description="No graph and no OD edge features. Learns only origin/destination FAF embeddings with year and fallback prior.",
    )

    # Controls for node features and shortcuts.
    specs["hgt_self_loop_only"] = hgt_spec(
        "hgt_self_loop_only",
        "HGT with only self-loop edges. Controls for HGT parameters without neighbor transfer.",
    )
    specs["hgt_no_zone_numeric_no_degree_features"] = hgt_spec(
        "hgt_no_zone_numeric_no_degree_features",
        "HGT full graph after removing zone_numeric and train-degree/count node features.",
        edge_variant="hgt_no_zone_numeric_no_degree_features",
        node_feature_variant="no_zone_numeric_no_degree",
    )
    specs["hgt_constant_node_features_full_graph"] = hgt_spec(
        "hgt_constant_node_features_full_graph",
        "HGT full graph with constant node features. Tests graph topology plus OD edge features without node attributes.",
        edge_variant="hgt_constant_node_features_full_graph",
        node_feature_variant="constant",
    )

    # Train-OD and demand-edge source controls.
    specs["hgt_train_od_only"] = hgt_spec(
        "hgt_train_od_only",
        "HGT using observed training OD graph edges, reverse edges, and self-loops only.",
    )
    specs["hgt_train_od_edges_removed_but_demand_kept"] = hgt_spec(
        "hgt_train_od_edges_removed_but_demand_kept",
        "HGT full graph with train_od and reverse train_od edges removed, retaining demand-mode edges and self-loops.",
    )
    specs["hgt_demand_edges_only"] = hgt_spec(
        "hgt_demand_edges_only",
        "HGT using all demand-mode edges and self-loops, without train_od edges.",
    )

    # Demand-mode split controls.
    mode_descriptions = {
        "hgt_truck_demand_only": "HGT using truck demand edges plus self-loops only.",
        "hgt_rail_demand_only": "HGT using rail demand edges plus self-loops only.",
        "hgt_multimodal_demand_only": "HGT using multimodal demand edges plus self-loops only.",
        "hgt_truck_plus_rail_demand": "HGT using truck and rail demand edges plus self-loops.",
        "hgt_truck_plus_multimodal_demand": "HGT using truck and multimodal demand edges plus self-loops.",
        "hgt_rail_plus_multimodal_demand": "HGT using rail and multimodal demand edges plus self-loops.",
    }
    for name, description in mode_descriptions.items():
        specs[name] = hgt_spec(name, description)

    # Multiple shuffled-edge perturbations.
    for replica_idx, perturb_seed in enumerate(cfg.graph_shuffle_seeds, start=1):
        name = f"hgt_shuffled_graph_edges_s{replica_idx}"
        specs[name] = hgt_spec(
            name,
            f"HGT full graph with non-self edge endpoints shuffled using perturbation seed {perturb_seed}.",
            edge_variant=name,
            perturbation_seed=perturb_seed,
        )

    # Multiple randomized-edge-type perturbations.
    for replica_idx, perturb_seed in enumerate(cfg.edge_type_randomization_seeds, start=1):
        name = f"hgt_randomized_edge_types_s{replica_idx}"
        specs[name] = hgt_spec(
            name,
            f"HGT full graph with non-self relation types randomized using perturbation seed {perturb_seed}.",
            edge_variant=name,
            perturbation_seed=perturb_seed,
        )

    return specs


all_ablation_specs = build_ablation_specs()
enabled_specs = [all_ablation_specs[name] for name in cfg.enabled_ablation_names]
print("Enabled ablation specifications:")
for spec in enabled_specs:
    print(f"- {spec.name}: {spec.model_type} | node_features={spec.node_feature_variant} | {spec.description}")


Enabled ablation specifications:
- hgt_full_faf_zone_graph: hgt | node_features=standard | HGT with train_od, all demand-mode edges, reverse edges, and self-loops.
- graphsage_same_edges: graphsage | node_features=standard | Homogeneous GraphSAGE using the same full FAF-zone graph as the full HGT variant.
- mlp_same_features_no_graph: mlp | node_features=standard | No-graph MLP residual model using OD edge features and fallback prior only.
- node_features_only_edge_decoder: node_feature_decoder | node_features=standard | No message passing. Uses origin node features, destination node features, OD edge features, and fallback prior.
- node_id_embedding_only: node_id_embedding | node_features=standard | No graph and no OD edge features. Learns only origin/destination FAF embeddings with year and fallback prior.
- hgt_self_loop_only: hgt | node_features=standard | HGT with only self-loop edges. Controls for HGT parameters without neighbor transfer.
- hgt_no_zone_numeric_no_degree_features:

## 17. Training and prediction utilities

This cell contains generic functions for GraphSAGE, HGT, and MLP ablation training. All ablations share the same loss, labels, edge features, fallback prior, and checkpoint selection criteria.

In [17]:
def make_batches(indices: torch.Tensor, batch_size: int, shuffle: bool) -> Iterable[torch.Tensor]:
    """Yield mini-batches of original dataframe row indices."""
    if shuffle:
        perm = indices[torch.randperm(indices.numel(), device=indices.device)]
    else:
        perm = indices
    for start in range(0, perm.numel(), batch_size):
        yield perm[start:start + batch_size]


def move_typed_edges(edge_dict_cpu: Mapping[Tuple[str, str, str], torch.Tensor]) -> Dict[Tuple[str, str, str], torch.Tensor]:
    """Move a typed edge dictionary to the active device."""
    return {key: value.to(device) for key, value in edge_dict_cpu.items()}


def get_node_features_for_spec(spec: AblationSpec) -> torch.Tensor:
    """Return the node feature tensor for one ablation specification."""
    if spec.node_feature_variant not in node_x_variants:
        raise KeyError(f"Unknown node feature variant: {spec.node_feature_variant}")
    return node_x_variants[spec.node_feature_variant]


def instantiate_model_for_spec(spec: AblationSpec) -> nn.Module:
    """Instantiate a model for one ablation specification."""
    year_count = len(year_to_idx)
    node_features_for_spec = get_node_features_for_spec(spec)
    node_in_dim = int(node_features_for_spec.shape[1])

    if spec.model_type == "hgt":
        metadata = spec.metadata()
        if metadata is None:
            raise ValueError(f"HGT spec {spec.name} has no metadata.")
        return HGTResidualQuantileModel(
            node_in_dim=node_in_dim,
            edge_dim=edge_attr.shape[1],
            year_count=year_count,
            metadata=metadata,
            config=cfg,
        )
    if spec.model_type == "graphsage":
        return GraphSAGEResidualQuantileModel(
            node_in_dim=node_in_dim,
            edge_dim=edge_attr.shape[1],
            year_count=year_count,
            config=cfg,
        )
    if spec.model_type == "node_feature_decoder":
        return NodeFeatureEdgeDecoderResidualQuantileModel(
            node_in_dim=node_in_dim,
            edge_dim=edge_attr.shape[1],
            year_count=year_count,
            config=cfg,
        )
    if spec.model_type == "node_id_embedding":
        return NodeIdEmbeddingOnlyResidualQuantileModel(
            n_nodes=len(zone_to_idx),
            year_count=year_count,
            config=cfg,
        )
    if spec.model_type == "mlp":
        return MLPResidualQuantileModel(
            edge_dim=edge_attr.shape[1],
            year_count=year_count,
            config=cfg,
        )
    raise ValueError(f"Unsupported model type: {spec.model_type}")


def forward_model(
    model: nn.Module,
    spec: AblationSpec,
    batch_indices: torch.Tensor,
    typed_edges_device: Optional[Mapping[Tuple[str, str, str], torch.Tensor]] = None,
    hom_edges_device: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Forward pass for one ablation model and a batch of supervised OD-year rows."""
    node_features_for_spec = get_node_features_for_spec(spec)

    if spec.model_type == "hgt":
        if typed_edges_device is None:
            raise ValueError("HGT forward requires typed edge dictionary.")
        return model(
            x_dict={"faf": node_features_for_spec},
            edge_index_dict=typed_edges_device,
            edge_label_index_batch=edge_label_index[:, batch_indices],
            edge_attr_batch=edge_attr[batch_indices],
            base_scaled_batch=base_prior_scaled[batch_indices],
            year_idx_batch=year_idx[batch_indices],
        )
    if spec.model_type == "graphsage":
        if hom_edges_device is None:
            raise ValueError("GraphSAGE forward requires homogeneous edge index.")
        return model(
            node_features=node_features_for_spec,
            edge_index=hom_edges_device,
            edge_label_index_batch=edge_label_index[:, batch_indices],
            edge_attr_batch=edge_attr[batch_indices],
            base_scaled_batch=base_prior_scaled[batch_indices],
            year_idx_batch=year_idx[batch_indices],
        )
    if spec.model_type == "node_feature_decoder":
        return model(
            node_features=node_features_for_spec,
            edge_label_index_batch=edge_label_index[:, batch_indices],
            edge_attr_batch=edge_attr[batch_indices],
            base_scaled_batch=base_prior_scaled[batch_indices],
            year_idx_batch=year_idx[batch_indices],
        )
    if spec.model_type == "node_id_embedding":
        return model(
            edge_label_index_batch=edge_label_index[:, batch_indices],
            base_scaled_batch=base_prior_scaled[batch_indices],
            year_idx_batch=year_idx[batch_indices],
        )
    if spec.model_type == "mlp":
        return model(
            edge_attr_batch=edge_attr[batch_indices],
            base_scaled_batch=base_prior_scaled[batch_indices],
            year_idx_batch=year_idx[batch_indices],
        )
    raise ValueError(f"Unsupported model type: {spec.model_type}")


def predict_model(
    model: nn.Module,
    spec: AblationSpec,
    indices: torch.Tensor,
    typed_edges_device: Optional[Mapping[Tuple[str, str, str], torch.Tensor]] = None,
    hom_edges_device: Optional[torch.Tensor] = None,
    batch_size: int = 16384,
) -> np.ndarray:
    """Predict q25/q50/q75 in minutes for selected rows."""
    model.eval()
    outputs: List[np.ndarray] = []
    with torch.no_grad():
        for batch in make_batches(indices, batch_size=batch_size, shuffle=False):
            pred_scaled = forward_model(model, spec, batch, typed_edges_device, hom_edges_device)
            pred = pred_scaled.detach().cpu().numpy() * target_scale
            outputs.append(pred.astype(np.float32))
    return np.concatenate(outputs, axis=0) if outputs else np.zeros((0, 3), dtype=np.float32)


def evaluate_model(model: nn.Module, spec: AblationSpec, indices: torch.Tensor, typed_edges_device: Optional[Mapping[Tuple[str, str, str], torch.Tensor]] = None, hom_edges_device: Optional[torch.Tensor] = None) -> Dict[str, float]:
    """Evaluate model on selected row indices."""
    pred = predict_model(model, spec, indices, typed_edges_device, hom_edges_device)
    idx_np = indices.detach().cpu().numpy()
    return compute_metrics_for_indices(pred, idx_np)


def state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    """Copy a state dict to CPU tensors."""
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}


@dataclass
class AblationTrainResult:
    """Container for one ablation/seed training result."""
    ablation_name: str
    model_type: str
    node_feature_variant: str
    perturbation_seed: Optional[int]
    seed: int
    description: str
    best_states: Dict[str, Dict[str, torch.Tensor]]
    best_records: Dict[str, Dict[str, float]]
    history: pd.DataFrame


def train_one_ablation(spec: AblationSpec, seed: int) -> AblationTrainResult:
    """Train one ablation specification for one seed."""
    set_global_seed(seed)
    model = instantiate_model_for_spec(spec).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)

    typed_edges_device = move_typed_edges(spec.typed_edge_dict_cpu) if spec.typed_edge_dict_cpu is not None else None
    hom_edges_device = spec.hom_edge_index_cpu.to(device) if spec.hom_edge_index_cpu is not None else None

    best_weighted_pinball = math.inf
    epochs_without_improvement = 0
    best_values = {"best_val_pinball": math.inf, "best_val_q75": math.inf, "best_val_iqr": math.inf}
    best_states: Dict[str, Dict[str, torch.Tensor]] = {}
    best_records: Dict[str, Dict[str, float]] = {}
    history_rows: List[Dict[str, Any]] = []

    start_time = time.time()
    for epoch in range(1, cfg.max_epochs + 1):
        model.train()
        batch_losses: List[float] = []
        for batch in make_batches(split_indices["train"], cfg.batch_size, shuffle=True):
            pred_scaled = forward_model(model, spec, batch, typed_edges_device, hom_edges_device)
            loss = training_loss(pred_scaled, y_scaled[batch], weights[batch], cfg.lambda_iqr)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            if cfg.grad_clip_norm > 0:
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
            optimizer.step()
            batch_losses.append(float(loss.detach().cpu()))

        if epoch % cfg.eval_every == 0:
            val_metrics = evaluate_model(model, spec, split_indices["val"], typed_edges_device, hom_edges_device)
            train_loss = float(np.mean(batch_losses)) if batch_losses else np.nan
            record = {
                "ablation_name": spec.name,
                "model_type": spec.model_type,
                "node_feature_variant": spec.node_feature_variant,
                "perturbation_seed": spec.perturbation_seed,
                "seed": seed,
                "epoch": epoch,
                "train_loss_scaled": train_loss,
                "elapsed_seconds": time.time() - start_time,
                **{f"val_{key}": value for key, value in val_metrics.items()},
            }
            history_rows.append(record)

            selector_values = {
                "best_val_pinball": val_metrics["pinball_mean"],
                "best_val_q75": val_metrics["mae_q75"],
                "best_val_iqr": val_metrics["iqr_mae"],
            }
            for selector_name, selector_value in selector_values.items():
                if selector_value < best_values[selector_name]:
                    best_values[selector_name] = float(selector_value)
                    best_states[selector_name] = state_dict_to_cpu(model)
                    best_records[selector_name] = {
                        "ablation_name": spec.name,
                        "model_type": spec.model_type,
                        "node_feature_variant": spec.node_feature_variant,
                        "perturbation_seed": spec.perturbation_seed,
                        "seed": seed,
                        "checkpoint_metric": selector_name,
                        "best_epoch": epoch,
                        "selector_value": float(selector_value),
                        "val_pinball_mean": float(val_metrics["pinball_mean"]),
                        "val_weighted_pinball_mean": float(val_metrics["weighted_pinball_mean"]),
                        "val_mae_q75": float(val_metrics["mae_q75"]),
                        "val_iqr_mae": float(val_metrics["iqr_mae"]),
                    }

            current_weighted = float(val_metrics["weighted_pinball_mean"])
            if current_weighted < best_weighted_pinball:
                best_weighted_pinball = current_weighted
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += cfg.eval_every

            if epoch == 1 or epoch % 25 == 0:
                print(
                    f"[{spec.name} | seed={seed}] epoch={epoch:04d} "
                    f"loss={train_loss:.5f} val_pinball={val_metrics['pinball_mean']:.3f} "
                    f"val_q75={val_metrics['mae_q75']:.3f} val_iqr={val_metrics['iqr_mae']:.3f}"
                )

            if epochs_without_improvement >= cfg.patience:
                print(f"Early stopping {spec.name} seed={seed} at epoch {epoch}.")
                break

    for selector_name in CHECKPOINT_METRICS:
        if selector_name not in best_states:
            best_states[selector_name] = state_dict_to_cpu(model)
            best_records[selector_name] = {
                "ablation_name": spec.name,
                "model_type": spec.model_type,
                "node_feature_variant": spec.node_feature_variant,
                "perturbation_seed": spec.perturbation_seed,
                "seed": seed,
                "checkpoint_metric": selector_name,
                "best_epoch": epoch,
                "selector_value": np.nan,
            }

    return AblationTrainResult(
        ablation_name=spec.name,
        model_type=spec.model_type,
        node_feature_variant=spec.node_feature_variant,
        perturbation_seed=spec.perturbation_seed,
        seed=seed,
        description=spec.description,
        best_states=best_states,
        best_records=best_records,
        history=pd.DataFrame(history_rows),
    )


## 18. Run the ablation suite

This cell trains all enabled ablations across all configured seeds. It also saves the requested validation-selected checkpoints.

In [18]:
train_results: List[AblationTrainResult] = []
checkpoint_records: List[Dict[str, Any]] = []

for spec in enabled_specs:
    for seed in cfg.seeds:
        result = train_one_ablation(spec, seed)
        train_results.append(result)

# Save checkpoint files and build checkpoint summary rows.
for result in train_results:
    spec = all_ablation_specs[result.ablation_name]
    seed_dir = models_dir / result.ablation_name / f"seed_{result.seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    for checkpoint_metric, state in result.best_states.items():
        record = result.best_records.get(checkpoint_metric, {}).copy()
        checkpoint_path = seed_dir / f"{checkpoint_metric}.pt"
        if cfg.save_model_checkpoints:
            torch.save(
                {
                    "state_dict": state,
                    "ablation_name": result.ablation_name,
                    "model_type": result.model_type,
                    "node_feature_variant": result.node_feature_variant,
                    "perturbation_seed": result.perturbation_seed,
                    "seed": result.seed,
                    "checkpoint_metric": checkpoint_metric,
                    "description": result.description,
                    "config": asdict(cfg),
                    "node_feature_columns_by_variant": node_feature_columns_by_variant,
                    "edge_feature_columns": edge_feature_columns,
                    "target_scale": target_scale,
                    "year_to_idx": year_to_idx,
                    "best_record": record,
                },
                checkpoint_path,
            )
        record["checkpoint_path"] = str(checkpoint_path)
        record["description"] = result.description
        checkpoint_records.append(record)

training_history = pd.concat([result.history for result in train_results], ignore_index=True) if train_results else pd.DataFrame()
checkpoint_summary = pd.DataFrame(checkpoint_records)

print("Training history rows:", len(training_history))
print("Checkpoint summary rows:", len(checkpoint_summary))
if not checkpoint_summary.empty:
    print(checkpoint_summary[["ablation_name", "node_feature_variant", "seed", "checkpoint_metric", "best_epoch", "selector_value"]].head(30))


[hgt_full_faf_zone_graph | seed=7] epoch=0001 loss=0.20002 val_pinball=459.338 val_q75=1314.362 val_iqr=797.810
[hgt_full_faf_zone_graph | seed=7] epoch=0025 loss=0.06348 val_pinball=178.560 val_q75=656.335 val_iqr=554.772
[hgt_full_faf_zone_graph | seed=7] epoch=0050 loss=0.04615 val_pinball=140.236 val_q75=512.747 val_iqr=448.202
[hgt_full_faf_zone_graph | seed=7] epoch=0075 loss=0.03912 val_pinball=137.837 val_q75=504.377 val_iqr=428.737
[hgt_full_faf_zone_graph | seed=7] epoch=0100 loss=0.03534 val_pinball=133.292 val_q75=488.352 val_iqr=412.536
[hgt_full_faf_zone_graph | seed=7] epoch=0125 loss=0.03292 val_pinball=130.802 val_q75=471.985 val_iqr=397.425
[hgt_full_faf_zone_graph | seed=7] epoch=0150 loss=0.03180 val_pinball=126.338 val_q75=466.872 val_iqr=400.369
[hgt_full_faf_zone_graph | seed=7] epoch=0175 loss=0.03008 val_pinball=122.498 val_q75=447.558 val_iqr=386.593
[hgt_full_faf_zone_graph | seed=7] epoch=0200 loss=0.02902 val_pinball=126.147 val_q75=458.504 val_iqr=388.210


## 19. Build prediction tables for every seed and checkpoint

Predictions are stored for validation and test splits. These tables support seed summaries, seed ensembles, strict test-only segment summaries, and paired comparisons.

In [19]:
def get_spec_by_name(name: str) -> AblationSpec:
    """Return ablation specification by name."""
    if name not in all_ablation_specs:
        raise KeyError(f"Unknown ablation spec: {name}")
    return all_ablation_specs[name]


def build_prediction_frame(result: AblationTrainResult, checkpoint_metric: str, state_dict: Mapping[str, torch.Tensor], splits: Sequence[str] = ("val", "test")) -> pd.DataFrame:
    """Build a normalized prediction dataframe for one trained ablation checkpoint."""
    spec = get_spec_by_name(result.ablation_name)
    model = instantiate_model_for_spec(spec).to(device)
    model.load_state_dict({key: value.to(device) for key, value in state_dict.items()})
    model.eval()

    typed_edges_device = move_typed_edges(spec.typed_edge_dict_cpu) if spec.typed_edge_dict_cpu is not None else None
    hom_edges_device = spec.hom_edge_index_cpu.to(device) if spec.hom_edge_index_cpu is not None else None

    frames: List[pd.DataFrame] = []
    for split_name in splits:
        indices = split_indices[split_name]
        idx_np = indices.detach().cpu().numpy()
        rows = df.iloc[idx_np].copy()
        pred = predict_model(model, spec, indices, typed_edges_device, hom_edges_device)
        frame = pd.DataFrame({
            "source": "ABLATION_V2_CONTROLS",
            "model": result.ablation_name,
            "model_type": result.model_type,
            "node_feature_variant": result.node_feature_variant,
            "perturbation_seed": result.perturbation_seed,
            "variant": "postprocessed",
            "checkpoint_metric": checkpoint_metric,
            "seed": int(result.seed),
            "split": split_name,
            origin_col: rows[origin_col].to_numpy(),
            dest_col: rows[dest_col].to_numpy(),
            year_col: rows[year_col].to_numpy(dtype=np.int64),
            "od_pair": rows["od_pair"].to_numpy(),
            "true_q25": rows["truck_q25"].to_numpy(dtype=np.float32),
            "true_q50": rows["truck_q50"].to_numpy(dtype=np.float32),
            "true_q75": rows["truck_q75"].to_numpy(dtype=np.float32),
            "pred_q25": pred[:, 0],
            "pred_q50": pred[:, 1],
            "pred_q75": pred[:, 2],
            "sample_weight": rows["sample_weight"].to_numpy(dtype=np.float32),
        })
        if "n_fmi_county_pairs" in rows.columns:
            frame["n_fmi_county_pairs"] = pd.to_numeric(rows["n_fmi_county_pairs"], errors="coerce").to_numpy(dtype=np.float32)
        else:
            frame["n_fmi_county_pairs"] = np.nan
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


prediction_frames: List[pd.DataFrame] = []
for result in train_results:
    for checkpoint_metric, state in result.best_states.items():
        prediction_frames.append(build_prediction_frame(result, checkpoint_metric, state))

ablation_predictions_by_seed = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
print("Ablation by-seed prediction shape:", ablation_predictions_by_seed.shape)
if not ablation_predictions_by_seed.empty:
    print(ablation_predictions_by_seed[["source", "model", "node_feature_variant", "seed", "checkpoint_metric", "split"]].drop_duplicates().head(30))


Ablation by-seed prediction shape: (815670, 21)
                     source                    model node_feature_variant  \
0      ABLATION_V2_CONTROLS  hgt_full_faf_zone_graph             standard   
957    ABLATION_V2_CONTROLS  hgt_full_faf_zone_graph             standard   
2014   ABLATION_V2_CONTROLS  hgt_full_faf_zone_graph             standard   
2971   ABLATION_V2_CONTROLS  hgt_full_faf_zone_graph             standard   
4028   ABLATION_V2_CONTROLS  hgt_full_faf_zone_graph             standard   
4985   ABLATION_V2_CONTROLS  hgt_full_faf_zone_graph             standard   
6042   ABLATION_V2_CONTROLS  hgt_full_faf_zone_graph             standard   
6999   ABLATION_V2_CONTROLS  hgt_full_faf_zone_graph             standard   
8056   ABLATION_V2_CONTROLS  hgt_full_faf_zone_graph             standard   
9013   ABLATION_V2_CONTROLS  hgt_full_faf_zone_graph             standard   
10070  ABLATION_V2_CONTROLS  hgt_full_faf_zone_graph             standard   
11027  ABLATION_V2_CONTROLS 

## 20. Add row-level errors and build seed-ensemble predictions

Seed ensembles average predictions across the configured seeds for the same ablation and checkpoint. They are useful as practical models but should be reported alongside single-seed mean/std results.

In [20]:
def add_error_columns(pred_df: pd.DataFrame) -> pd.DataFrame:
    """Add row-level error and pinball columns to a prediction dataframe."""
    result = pred_df.copy()
    y_true_local = result[["true_q25", "true_q50", "true_q75"]].to_numpy(dtype=np.float32)
    y_pred_local = result[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(dtype=np.float32)
    abs_error = np.abs(y_pred_local - y_true_local)
    pin = pinball_numpy(y_true_local, y_pred_local)
    result["error_q25"] = y_pred_local[:, 0] - y_true_local[:, 0]
    result["error_q50"] = y_pred_local[:, 1] - y_true_local[:, 1]
    result["error_q75"] = y_pred_local[:, 2] - y_true_local[:, 2]
    result["abs_error_q25"] = abs_error[:, 0]
    result["abs_error_q50"] = abs_error[:, 1]
    result["abs_error_q75"] = abs_error[:, 2]
    result["true_iqr"] = y_true_local[:, 2] - y_true_local[:, 0]
    result["pred_iqr"] = y_pred_local[:, 2] - y_pred_local[:, 0]
    result["abs_error_iqr"] = np.abs(result["pred_iqr"] - result["true_iqr"])
    result["pinball_q25"] = pin[:, 0]
    result["pinball_q50"] = pin[:, 1]
    result["pinball_q75"] = pin[:, 2]
    result["pinball_mean_row"] = pin.mean(axis=1)
    return result


def build_seed_ensemble(pred_df: pd.DataFrame) -> pd.DataFrame:
    """Average predictions over seeds for each ablation/checkpoint/split/OD-year row."""
    if pred_df.empty:
        return pred_df.copy()
    key_cols = ["source", "model", "model_type", "node_feature_variant", "perturbation_seed", "variant", "checkpoint_metric", "split", origin_col, dest_col, year_col, "od_pair"]
    value_first_cols = ["true_q25", "true_q50", "true_q75", "sample_weight", "n_fmi_county_pairs"]
    grouped = pred_df.groupby(key_cols, dropna=False)
    pred_mean = grouped[["pred_q25", "pred_q50", "pred_q75"]].mean().reset_index()
    first_values = grouped[value_first_cols].first().reset_index()
    merged = pred_mean.merge(first_values, on=key_cols, how="left")
    merged["source"] = "ABLATION_V2_ENSEMBLE"
    merged["seed"] = "ensemble"
    return merged


ablation_predictions_by_seed = add_error_columns(ablation_predictions_by_seed)
ablation_predictions_seed_ensemble = add_error_columns(build_seed_ensemble(ablation_predictions_by_seed)) if cfg.build_seed_ensembles else pd.DataFrame()
print("Seed-ensemble prediction shape:", ablation_predictions_seed_ensemble.shape)

Seed-ensemble prediction shape: (163134, 34)


## 21. Load previous Cold-OD non-graph baseline predictions

The ablation results are combined with the previous Cold-OD non-graph baselines for context. This includes the current strongest non-graph baseline, `mlp_residual_prior_features`.

In [21]:
def normalize_baseline_prediction_schema(path: Path) -> pd.DataFrame:
    """Load and normalize a previous Cold-OD baseline prediction parquet."""
    if not path.exists():
        print(f"Cold-OD baseline prediction file not found: {path}")
        return pd.DataFrame()

    baseline = pd.read_parquet(path)
    baseline = normalize_prediction_key_columns(baseline, origin_col, dest_col, year_col)

    rename_map = {}
    for src, dst in [
        ("y_q25", "true_q25"),
        ("y_q50", "true_q50"),
        ("y_q75", "true_q75"),
        ("truck_q25", "true_q25"),
        ("truck_q50", "true_q50"),
        ("truck_q75", "true_q75"),
        ("pred_truck_q25", "pred_q25"),
        ("pred_truck_q50", "pred_q50"),
        ("pred_truck_q75", "pred_q75"),
    ]:
        if src in baseline.columns and dst not in baseline.columns:
            rename_map[src] = dst
    baseline = baseline.rename(columns=rename_map)

    ensure_columns(baseline, ["model", "split", "true_q25", "true_q50", "true_q75", "pred_q25", "pred_q50", "pred_q75"], "baseline predictions")
    if "source" not in baseline.columns:
        baseline["source"] = "ColdOD"
    else:
        baseline["source"] = baseline["source"].fillna("ColdOD").astype(str)
    baseline.loc[~baseline["source"].astype(str).str.contains("ColdOD", case=False, na=False), "source"] = "ColdOD"
    if "variant" not in baseline.columns:
        baseline["variant"] = "postprocessed"
    if "checkpoint_metric" not in baseline.columns:
        baseline["checkpoint_metric"] = "baseline"
    if "seed" not in baseline.columns:
        baseline["seed"] = "baseline"
    if "model_type" not in baseline.columns:
        baseline["model_type"] = "non_graph"
    if "sample_weight" not in baseline.columns:
        key_cols = [origin_col, dest_col, year_col]
        baseline = baseline.merge(df[key_cols + ["sample_weight"]], on=key_cols, how="left")
    if "n_fmi_county_pairs" not in baseline.columns:
        key_cols = [origin_col, dest_col, year_col]
        if "n_fmi_county_pairs" in df.columns:
            baseline = baseline.merge(df[key_cols + ["n_fmi_county_pairs"]], on=key_cols, how="left")
        else:
            baseline["n_fmi_county_pairs"] = np.nan
    return add_error_columns(baseline)


baseline_predictions = normalize_baseline_prediction_schema(cold_baseline_predictions_path)
combined_predictions = pd.concat(
    [frame for frame in [baseline_predictions, ablation_predictions_by_seed, ablation_predictions_seed_ensemble] if not frame.empty],
    ignore_index=True,
)
print("Baseline prediction shape:", baseline_predictions.shape)
print("Combined prediction shape:", combined_predictions.shape)
print(combined_predictions[["source", "model", "checkpoint_metric", "seed", "split"]].drop_duplicates().head(30))

Baseline prediction shape: (22154, 34)
Combined prediction shape: (1000958, 36)
                     source                                model  \
0                    ColdOD                  global_train_median   
957                  ColdOD                  global_train_median   
2014                 ColdOD                         origin_prior   
2971                 ColdOD                         origin_prior   
4028                 ColdOD                    destination_prior   
4985                 ColdOD                    destination_prior   
6042                 ColdOD       origin_destination_blend_prior   
6999                 ColdOD       origin_destination_blend_prior   
8056                 ColdOD     lightgbm_direct_current_features   
9013                 ColdOD     lightgbm_direct_current_features   
10070                ColdOD            prior_feature_lgbm_direct   
11027                ColdOD            prior_feature_lgbm_direct   
12084                ColdOD       re

## 22. Compute metrics, leaderboards, and seed summaries

Metrics are recomputed from prediction parquet tables rather than trusting previous CSV files. The leaderboard is restricted to the Cold-OD test split.

In [22]:
def compute_metrics_for_prediction_frame(pred_df: pd.DataFrame) -> pd.DataFrame:
    """Compute metrics by source/model/checkpoint/seed/split."""
    if pred_df.empty:
        return pd.DataFrame()
    group_cols = ["source", "model", "model_type", "node_feature_variant", "perturbation_seed", "variant", "checkpoint_metric", "seed", "split"]
    rows: List[Dict[str, Any]] = []
    for group_key, group in pred_df.groupby(group_cols, dropna=False):
        y_true_local = group[["true_q25", "true_q50", "true_q75"]].to_numpy(dtype=np.float32)
        y_pred_local = group[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(dtype=np.float32)
        row_weights = group["sample_weight"].to_numpy(dtype=np.float32) if "sample_weight" in group.columns else None
        row = dict(zip(group_cols, group_key))
        row.update(compute_metrics_from_arrays(y_true_local, y_pred_local, row_weights=row_weights))
        rows.append(row)
    return pd.DataFrame(rows)


metrics_by_seed = compute_metrics_for_prediction_frame(ablation_predictions_by_seed)
metrics_seed_ensemble = compute_metrics_for_prediction_frame(ablation_predictions_seed_ensemble)
combined_metrics = compute_metrics_for_prediction_frame(combined_predictions)
leaderboard_test = combined_metrics[combined_metrics["split"].eq("test")].sort_values("pinball_mean", ascending=True).reset_index(drop=True)
leaderboard_test.insert(0, "rank", np.arange(1, len(leaderboard_test) + 1))

# Mean/std only over individual seed rows, not seed-ensemble rows.
seed_metric_cols = [
    "pinball_mean", "weighted_pinball_mean", "mae_q50", "mae_q75", "iqr_mae",
    "stress_top10_mae_q75", "top_iqr10_mae_q75",
]
individual_seed_metrics = metrics_by_seed[metrics_by_seed["split"].eq("test")].copy()
summary_rows: List[Dict[str, Any]] = []
for group_key, group in individual_seed_metrics.groupby(["source", "model", "model_type", "node_feature_variant", "perturbation_seed", "checkpoint_metric"], dropna=False):
    row = dict(zip(["source", "model", "model_type", "node_feature_variant", "perturbation_seed", "checkpoint_metric"], group_key))
    row["n_seeds"] = int(group["seed"].nunique())
    for metric in seed_metric_cols:
        row[f"{metric}_mean"] = float(group[metric].mean())
        row[f"{metric}_std"] = float(group[metric].std(ddof=1)) if len(group) > 1 else 0.0
    summary_rows.append(row)
seed_summary = pd.DataFrame(summary_rows).sort_values("pinball_mean_mean", ascending=True).reset_index(drop=True)

print("Test leaderboard top 20:")
print(leaderboard_test[["rank", "source", "model", "checkpoint_metric", "seed", "pinball_mean", "mae_q75", "iqr_mae"]].head(20).to_string(index=False))
print("\nSeed summary top rows:")
print(seed_summary.head(20).to_string(index=False))

Test leaderboard top 20:
 rank               source                                  model checkpoint_metric     seed  pinball_mean    mae_q75    iqr_mae
    1 ABLATION_V2_CONTROLS             hgt_multimodal_demand_only  best_val_pinball      123    129.690155 457.936676 376.754303
    2 ABLATION_V2_ENSEMBLE            hgt_shuffled_graph_edges_s1      best_val_iqr ensemble    130.193375 442.230713 368.211823
    3 ABLATION_V2_ENSEMBLE            hgt_shuffled_graph_edges_s1  best_val_pinball ensemble    130.251297 444.514069 372.478302
    4 ABLATION_V2_ENSEMBLE                   graphsage_same_edges  best_val_pinball ensemble    130.267502 440.057587 373.538910
    5 ABLATION_V2_ENSEMBLE                     hgt_self_loop_only  best_val_pinball ensemble    130.370346 441.030182 370.140503
    6 ABLATION_V2_ENSEMBLE                   graphsage_same_edges      best_val_iqr ensemble    130.449341 434.157349 366.421387
    7 ABLATION_V2_CONTROLS             hgt_multimodal_demand_only      b

## 23. Strict test-only segment summaries

This cell computes segment-level summaries only on the Cold-OD test rows. It verifies that every model/checkpoint/seed identity sums to exactly the number of Cold-OD test rows.

In [23]:
def assign_decile(series: pd.Series, name: str) -> pd.Series:
    """Assign stable decile labels to a numeric series."""
    ranks = series.rank(method="first")
    labels = [f"{name}_{idx:02d}" for idx in range(1, 11)]
    try:
        return pd.qcut(ranks, q=10, labels=labels, duplicates="drop").astype(str)
    except ValueError:
        return pd.Series([f"{name}_single_bin"] * len(series), index=series.index)


def assign_quartile(series: pd.Series, name: str) -> pd.Series:
    """Assign stable quartile labels to a numeric series."""
    ranks = series.rank(method="first")
    labels = [f"{name}_{idx:02d}" for idx in range(1, 5)]
    try:
        return pd.qcut(ranks, q=4, labels=labels, duplicates="drop").astype(str)
    except ValueError:
        return pd.Series([f"{name}_single_bin"] * len(series), index=series.index)


def add_test_segments(pred_df: pd.DataFrame) -> pd.DataFrame:
    """Add true-q75, true-IQR, and n_fmi segments to test-only prediction rows."""
    test_df = pred_df[pred_df["split"].eq("test")].copy()
    if test_df.empty:
        return test_df
    key_cols = [origin_col, dest_col, year_col]
    unique_test = test_df[key_cols + ["true_q25", "true_q50", "true_q75", "n_fmi_county_pairs"]].drop_duplicates(key_cols).copy()
    unique_test["true_iqr"] = unique_test["true_q75"] - unique_test["true_q25"]
    unique_test["true_q75_decile"] = assign_decile(unique_test["true_q75"], "true_q75_decile")
    unique_test["true_iqr_decile"] = assign_decile(unique_test["true_iqr"], "true_iqr_decile")
    if unique_test["n_fmi_county_pairs"].notna().any():
        fill_value = unique_test["n_fmi_county_pairs"].median()
        unique_test["n_fmi_county_pairs_quartile"] = assign_quartile(unique_test["n_fmi_county_pairs"].fillna(fill_value), "n_fmi_county_pairs_quartile")
    else:
        unique_test["n_fmi_county_pairs_quartile"] = "n_fmi_county_pairs_quartile_missing"
    return test_df.merge(unique_test[key_cols + ["true_q75_decile", "true_iqr_decile", "n_fmi_county_pairs_quartile"]], on=key_cols, how="left")


def segment_summary_test_only(pred_df: pd.DataFrame, segment_col: str) -> pd.DataFrame:
    """Compute segment metrics strictly on test rows."""
    test_df = add_test_segments(pred_df)
    if test_df.empty or segment_col not in test_df.columns:
        return pd.DataFrame()
    group_cols = ["source", "model", "model_type", "node_feature_variant", "perturbation_seed", "variant", "checkpoint_metric", "seed", segment_col]
    rows: List[Dict[str, Any]] = []
    for group_key, group in test_df.groupby(group_cols, dropna=False):
        y_true_local = group[["true_q25", "true_q50", "true_q75"]].to_numpy(dtype=np.float32)
        y_pred_local = group[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(dtype=np.float32)
        row_weights = group["sample_weight"].to_numpy(dtype=np.float32)
        row = dict(zip(group_cols, group_key))
        row.update(compute_metrics_from_arrays(y_true_local, y_pred_local, row_weights=row_weights))
        rows.append(row)
    summary = pd.DataFrame(rows)

    expected_n = int(df["cold_split"].eq("test").sum())
    identity_cols = ["source", "model", "model_type", "node_feature_variant", "perturbation_seed", "variant", "checkpoint_metric", "seed"]
    totals = summary.groupby(identity_cols, dropna=False)["n_rows"].sum()
    bad = totals[~np.isclose(totals, expected_n)]
    if not bad.empty:
        raise AssertionError(f"Segment summary is not strict test-only. Expected {expected_n}; bad totals: {bad.head().to_dict()}")
    return summary.sort_values(identity_cols + [segment_col]).reset_index(drop=True)


segment_tables: Dict[str, pd.DataFrame] = {}
for segment_col in ["true_q75_decile", "true_iqr_decile", "n_fmi_county_pairs_quartile"]:
    segment_tables[segment_col] = segment_summary_test_only(combined_predictions, segment_col)
    print(segment_col, "rows:", len(segment_tables[segment_col]))
    print(segment_tables[segment_col].head(3).to_string(index=False))

true_q75_decile rows: 4970
              source                model model_type node_feature_variant  perturbation_seed       variant checkpoint_metric seed    true_q75_decile  n_rows    mae_q25    mae_q50    mae_q75   rmse_q50  pinball_q25  pinball_q50  pinball_q75  pinball_mean  weighted_pinball_mean    iqr_mae   bias_q25  bias_q50   bias_q75  pred_iqr_mean  true_iqr_mean  raw_crossing_rate  q50_inside_pred_q25_q75_rate  stress_top10_threshold_q75  stress_top10_n_rows_q75  stress_top10_mae_q75  stress_top10_iqr_mae  top_iqr10_threshold  top_iqr10_n_rows  top_iqr10_mae_q75  top_iqr10_iqr_mae
ABLATION_V2_CONTROLS graphsage_same_edges  graphsage             standard                NaN postprocessed      best_val_iqr    7 true_q75_decile_01     106  60.443401 106.663574 238.651443 151.158691    20.831429    53.331787    81.730278     51.964493              52.386776 250.749542 -37.561089 63.433090 150.381760    1032.294922     844.352051                0.0                      0.990566  

## 24. Strict test-only paired comparisons

This cell compares each ablation model to two references:

1. `ColdOD::mlp_residual_prior_features`, the strongest non-graph Cold-OD baseline.
2. `ABLATION_ENSEMBLE::hgt_full_faf_zone_graph::best_val_pinball`, the full graph HGT reference.

Positive deltas mean the candidate is better than the reference on the same test row.

In [24]:
def prediction_identity(frame: pd.DataFrame) -> pd.Series:
    """Create a stable prediction identity string."""
    return (
        frame["source"].astype(str)
        + "::" + frame["model"].astype(str)
        + "::node=" + frame.get("node_feature_variant", pd.Series("NA", index=frame.index)).astype(str)
        + "::perturb=" + frame.get("perturbation_seed", pd.Series("NA", index=frame.index)).astype(str)
        + "::" + frame["checkpoint_metric"].astype(str)
        + "::seed=" + frame["seed"].astype(str)
    )


def make_reference_mask(test_df: pd.DataFrame, source: str, model: str, checkpoint_metric: Optional[str] = None, seed: Optional[str] = None) -> pd.Series:
    """Build a mask for selecting a reference prediction identity."""
    mask = test_df["source"].astype(str).eq(source) & test_df["model"].astype(str).eq(model)
    if checkpoint_metric is not None:
        mask &= test_df["checkpoint_metric"].astype(str).eq(checkpoint_metric)
    if seed is not None:
        mask &= test_df["seed"].astype(str).eq(seed)
    return mask


def make_paired_summary_for_reference(pred_df: pd.DataFrame, reference_source: str, reference_model: str, reference_checkpoint: Optional[str] = None, reference_seed: Optional[str] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Compute strict test-only paired comparisons against a selected reference identity."""
    test_df = pred_df[pred_df["split"].eq("test")].copy()
    if test_df.empty:
        return pd.DataFrame(), pd.DataFrame()
    test_df["identity"] = prediction_identity(test_df)
    key_cols = [origin_col, dest_col, year_col]
    reference_candidates = test_df[make_reference_mask(test_df, reference_source, reference_model, reference_checkpoint, reference_seed)].copy()
    if reference_candidates.empty:
        print(f"Reference not found: {reference_source}::{reference_model}::{reference_checkpoint}::{reference_seed}")
        return pd.DataFrame(), pd.DataFrame()

    reference_identity = sorted(reference_candidates["identity"].unique().tolist())[0]
    reference = reference_candidates[reference_candidates["identity"].eq(reference_identity)].copy()
    expected_n = int(df["cold_split"].eq("test").sum())
    if len(reference) != expected_n:
        raise AssertionError(f"Reference paired set should have {expected_n} rows, got {len(reference)}.")

    metric_cols = ["pinball_mean_row", "abs_error_q25", "abs_error_q50", "abs_error_q75", "abs_error_iqr"]
    reference_small = reference[key_cols + metric_cols].copy()
    candidate_df = test_df[test_df["identity"].ne(reference_identity)].copy()

    summary_rows: List[Dict[str, Any]] = []
    paired_frames: List[pd.DataFrame] = []
    for candidate_identity, candidate in candidate_df.groupby("identity"):
        merged = reference_small.merge(
            candidate[key_cols + metric_cols + ["source", "model", "model_type", "checkpoint_metric", "seed"]],
            on=key_cols,
            suffixes=("_reference", "_candidate"),
            how="inner",
        )
        if len(merged) != expected_n:
            raise AssertionError(f"Candidate {candidate_identity} paired rows should be {expected_n}, got {len(merged)}.")
        for metric in metric_cols:
            merged[f"delta_{metric}"] = merged[f"{metric}_reference"] - merged[f"{metric}_candidate"]
            merged[f"candidate_wins_{metric}"] = merged[f"delta_{metric}"] > 0
        candidate_info = candidate[["source", "model", "model_type", "checkpoint_metric", "seed"]].iloc[0].to_dict()
        row = {
            "reference_identity": reference_identity,
            "candidate_identity": candidate_identity,
            "n_rows": len(merged),
            **{f"candidate_{key}": value for key, value in candidate_info.items()},
        }
        for metric in metric_cols:
            row[f"mean_delta_{metric}"] = float(merged[f"delta_{metric}"].mean())
            row[f"median_delta_{metric}"] = float(merged[f"delta_{metric}"].median())
            row[f"win_rate_{metric}"] = float(merged[f"candidate_wins_{metric}"].mean())
        summary_rows.append(row)
        merged["reference_identity"] = reference_identity
        merged["candidate_identity"] = candidate_identity
        paired_frames.append(merged)

    summary = pd.DataFrame(summary_rows).sort_values("mean_delta_pinball_mean_row", ascending=False).reset_index(drop=True)
    paired_rows = pd.concat(paired_frames, ignore_index=True) if paired_frames else pd.DataFrame()
    return summary, paired_rows


paired_vs_mlp_summary, paired_vs_mlp_rows = make_paired_summary_for_reference(
    combined_predictions,
    reference_source="ColdOD",
    reference_model="mlp_residual_prior_features",
)

paired_vs_full_hgt_summary, paired_vs_full_hgt_rows = make_paired_summary_for_reference(
    combined_predictions,
    reference_source="ABLATION_V2_ENSEMBLE",
    reference_model="hgt_full_faf_zone_graph",
    reference_checkpoint="best_val_pinball",
    reference_seed="ensemble",
)

paired_summary = pd.concat([
    paired_vs_mlp_summary.assign(reference_name="ColdOD::mlp_residual_prior_features"),
    paired_vs_full_hgt_summary.assign(reference_name="ABLATION_V2_ENSEMBLE::hgt_full_faf_zone_graph::best_val_pinball"),
], ignore_index=True)
paired_rows = pd.concat([
    paired_vs_mlp_rows.assign(reference_name="ColdOD::mlp_residual_prior_features"),
    paired_vs_full_hgt_rows.assign(reference_name="ABLATION_V2_ENSEMBLE::hgt_full_faf_zone_graph::best_val_pinball"),
], ignore_index=True) if (not paired_vs_mlp_rows.empty or not paired_vs_full_hgt_rows.empty) else pd.DataFrame()

print("Paired summary rows:", len(paired_summary))
print(paired_summary.head(30).to_string(index=False))

Paired summary rows: 992
                                                                 reference_identity                                                                                                                      candidate_identity  n_rows     candidate_source                        candidate_model candidate_model_type candidate_checkpoint_metric candidate_seed  mean_delta_pinball_mean_row  median_delta_pinball_mean_row  win_rate_pinball_mean_row  mean_delta_abs_error_q25  median_delta_abs_error_q25  win_rate_abs_error_q25  mean_delta_abs_error_q50  median_delta_abs_error_q50  win_rate_abs_error_q50  mean_delta_abs_error_q75  median_delta_abs_error_q75  win_rate_abs_error_q75  mean_delta_abs_error_iqr  median_delta_abs_error_iqr  win_rate_abs_error_iqr                      reference_name
ColdOD::mlp_residual_prior_features::node=nan::perturb=nan::baseline::seed=baseline                               ABLATION_V2_CONTROLS::hgt_multimodal_demand_only::node=standard::perturb=N

## 25. Optional diagnostic plots

The plots are optional and are saved as PNG files. They are not required for the experiment to complete.

In [25]:
if cfg.make_plots:
    try:
        import matplotlib.pyplot as plt

        plot_df = leaderboard_test[leaderboard_test["source"].isin(["ColdOD", "ABLATION_ENSEMBLE"])].copy()
        plot_df["label"] = plot_df["source"].astype(str) + "::" + plot_df["model"].astype(str) + "::" + plot_df["checkpoint_metric"].astype(str)

        fig_df = plot_df.sort_values("pinball_mean", ascending=True).head(30)
        plt.figure(figsize=(12, max(5, 0.42 * len(fig_df))))
        plt.barh(fig_df["label"], fig_df["pinball_mean"])
        plt.gca().invert_yaxis()
        plt.xlabel("pinball_mean")
        plt.title("Cold-OD Graph Ablation Test Pinball Mean")
        plt.tight_layout()
        plt.savefig(plots_dir / "graph_ablation_leaderboard_pinball_mean.png", dpi=160)
        plt.close()

        q75_seg = segment_tables.get("true_q75_decile", pd.DataFrame()).copy()
        if not q75_seg.empty:
            selected = [
                ("ABLATION_ENSEMBLE", "hgt_full_faf_zone_graph"),
                ("ABLATION_ENSEMBLE", "hgt_train_od_only"),
                ("ABLATION_ENSEMBLE", "hgt_demand_edges_only"),
                ("ABLATION_ENSEMBLE", "hgt_self_loop_only"),
                ("ABLATION_ENSEMBLE", "graphsage_same_edges"),
                ("ColdOD", "mlp_residual_prior_features"),
            ]
            plt.figure(figsize=(14, 6))
            for source_name, model_name in selected:
                part = q75_seg[(q75_seg["source"].eq(source_name)) & (q75_seg["model"].eq(model_name))]
                if source_name == "ABLATION_ENSEMBLE":
                    part = part[part["checkpoint_metric"].eq("best_val_pinball")]
                if part.empty:
                    continue
                part = part.sort_values("true_q75_decile")
                plt.plot(part["true_q75_decile"], part["mae_q75"], marker="o", label=f"{source_name}::{model_name}")
            plt.xticks(rotation=45, ha="right")
            plt.ylabel("mae_q75")
            plt.xlabel("true_q75_decile")
            plt.title("Cold-OD q75 MAE by True q75 Decile: Graph Ablation")
            plt.legend()
            plt.tight_layout()
            plt.savefig(plots_dir / "graph_ablation_q75_mae_by_true_q75_decile.png", dpi=160)
            plt.close()

        print("Plots written to:", plots_dir)
    except Exception as exc:
        print("Plot generation skipped due to error:", repr(exc))

Plots written to: E:\NetworkOptimization\pythonProject1\Data\10_experiments\graph_cold_od_ablation_v2_controls_notebook\east_plus_gulf\plots


## 26. Save all artifacts with Parquet-safe schemas

This cell saves prediction tables, metrics, segment summaries, paired comparisons, and metadata. The Parquet writer normalizes mixed object columns, especially `seed`, to avoid PyArrow mixed-type failures.

In [26]:
FORCE_STRING_COLUMNS = {
    "source", "model", "model_type", "node_feature_variant", "variant", "checkpoint_metric", "seed", "split",
    "faf_orig", "faf_dest", "origin", "destination", "od_pair",
    "comparison", "reference_model", "candidate_model", "reference_identity", "candidate_identity", "perturbation_seed",
}


def json_default(value: Any) -> Any:
    """Convert NumPy, pandas, and Path objects to JSON-serializable objects."""
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return str(value)


def stringify_object_value(value: Any) -> Any:
    """Convert object scalar values to Parquet-safe strings or missing values."""
    if isinstance(value, (dict, list, tuple, set)):
        return json.dumps(value, sort_keys=True, default=json_default)
    if value is None:
        return pd.NA
    try:
        if pd.isna(value):
            return pd.NA
    except Exception:
        pass
    return str(value)


def make_unique_columns(columns: Sequence[Any]) -> List[str]:
    """Return unique string column names while preserving order."""
    seen: Dict[str, int] = {}
    result: List[str] = []
    for column in columns:
        base = str(column)
        count = seen.get(base, 0)
        result.append(base if count == 0 else f"{base}__dup{count}")
        seen[base] = count + 1
    return result


def normalize_dataframe_for_parquet(frame: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with Parquet-safe dtypes."""
    clean = frame.copy()
    clean.columns = make_unique_columns(clean.columns)
    for column in clean.columns:
        series = clean[column]
        is_object_like = pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series) or str(series.dtype) == "category"
        if column in FORCE_STRING_COLUMNS or is_object_like:
            clean[column] = series.map(stringify_object_value).astype("string")
    return clean


def safe_to_parquet(frame: pd.DataFrame, path: Path) -> Path:
    """Write a dataframe to parquet after dtype normalization."""
    path.parent.mkdir(parents=True, exist_ok=True)
    normalize_dataframe_for_parquet(frame).to_parquet(path, index=False, engine="pyarrow")
    return path


def write_json(payload: Dict[str, Any], path: Path) -> Path:
    """Write a JSON file with robust serialization."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, default=json_default)
    return path


# Prediction outputs.
ablation_by_seed_path = output_dir / "predictions_graph_cold_od_ablation_v2_controls_val_test_by_seed.parquet"
ablation_seed_ensemble_path = output_dir / "predictions_graph_cold_od_ablation_v2_controls_val_test_seed_ensemble.parquet"
combined_predictions_path = output_dir / "combined_predictions_graph_cold_od_ablation_v2_controls_val_test.parquet"

# Metric outputs.
metrics_by_seed_path = output_dir / "metrics_graph_cold_od_ablation_v2_controls_by_seed.csv"
metrics_seed_ensemble_path = output_dir / "metrics_graph_cold_od_ablation_v2_controls_seed_ensemble.csv"
combined_metrics_path = output_dir / "metrics_graph_cold_od_ablation_v2_controls_combined.csv"
leaderboard_path = output_dir / "leaderboard_test_graph_cold_od_ablation_v2_controls.csv"
seed_summary_path = output_dir / "seed_summary_graph_cold_od_ablation_v2_controls.csv"
checkpoint_summary_path = output_dir / "checkpoint_summary_graph_cold_od_ablation_v2_controls.csv"
history_path = output_dir / "training_history_graph_cold_od_ablation_v2_controls.csv"

# Diagnostics.
paired_summary_path = output_dir / "paired_summary_graph_cold_od_ablation_v2_controls_test_only.csv"
paired_rows_path = output_dir / "paired_rows_graph_cold_od_ablation_v2_controls_test_only.parquet"

safe_to_parquet(ablation_predictions_by_seed, ablation_by_seed_path)
safe_to_parquet(ablation_predictions_seed_ensemble, ablation_seed_ensemble_path)
safe_to_parquet(combined_predictions, combined_predictions_path)

metrics_by_seed.to_csv(metrics_by_seed_path, index=False)
metrics_seed_ensemble.to_csv(metrics_seed_ensemble_path, index=False)
combined_metrics.to_csv(combined_metrics_path, index=False)
leaderboard_test.to_csv(leaderboard_path, index=False)
seed_summary.to_csv(seed_summary_path, index=False)
checkpoint_summary.to_csv(checkpoint_summary_path, index=False)
training_history.to_csv(history_path, index=False)
paired_summary.to_csv(paired_summary_path, index=False)
if not paired_rows.empty:
    safe_to_parquet(paired_rows, paired_rows_path)

for segment_col, table in segment_tables.items():
    table.to_csv(tables_dir / f"graph_ablation_v2_controls_test_only_segment_summary__{segment_col}.csv", index=False)

split_summary.to_csv(tables_dir / "cold_od_split_summary_ablation_v2_controls.csv", index=False)
node_feature_df.to_csv(tables_dir / "faf_node_features_train_only_ablation_v2_controls.csv", index=False)
edge_count_table.to_csv(tables_dir / "graph_edge_counts_ablation_v2_controls.csv", index=False)

write_json({
    "manifest_feature_columns": manifest_feature_columns,
    "cold_prior_feature_columns": COLD_PRIOR_COLUMNS,
    "edge_feature_columns": edge_feature_columns,
    "node_feature_columns_by_variant": node_feature_columns_by_variant,
    "label_columns": LABEL_COLUMNS,
    "ablation_names": list(cfg.enabled_ablation_names),
    "graph_shuffle_seeds": list(cfg.graph_shuffle_seeds),
    "edge_type_randomization_seeds": list(cfg.edge_type_randomization_seeds),
}, output_dir / "feature_columns_graph_cold_od_ablation_v2_controls.json")

write_json({
    "edge_preprocessor": edge_preprocessor.to_dict(),
    "node_normalization_metadata": node_normalization_metadata,
    "target_scale": float(target_scale),
    "year_to_idx": {str(key): int(value) for key, value in year_to_idx.items()},
}, output_dir / "preprocessors_graph_cold_od_ablation_v2_controls.json")

run_config = {
    "notebook": "Graph_ColdOD_Ablation_v2_Controls",
    "config": asdict(cfg),
    "paths": {
        "supervised_path": str(supervised_path),
        "manifest_path": str(manifest_path),
        "cold_baseline_predictions_path": str(cold_baseline_predictions_path),
        "cold_baseline_all_splits_path": str(cold_baseline_all_splits_path),
        "output_dir": str(output_dir),
    },
    "dataset": {
        "n_rows": int(len(df)),
        "split_counts": {str(k): int(v) for k, v in df["cold_split"].value_counts().to_dict().items()},
        "origin_column": origin_col,
        "destination_column": dest_col,
        "year_column": year_col,
        "n_nodes": int(len(zone_to_idx)),
        "edge_feature_dim": int(edge_attr_cpu.shape[1]),
        "node_feature_dims": {name: int(tensor.shape[1]) for name, tensor in node_x_variants_cpu.items()},
    },
    "package_versions": {
        "python": os.sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "torch": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
    },
}
write_json(run_config, output_dir / "run_config_graph_cold_od_ablation_v2_controls.json")

artifact_manifest = {
    "ablation_by_seed_predictions": str(ablation_by_seed_path),
    "ablation_seed_ensemble_predictions": str(ablation_seed_ensemble_path),
    "combined_predictions": str(combined_predictions_path),
    "leaderboard_test": str(leaderboard_path),
    "combined_metrics": str(combined_metrics_path),
    "seed_summary": str(seed_summary_path),
    "checkpoint_summary": str(checkpoint_summary_path),
    "training_history": str(history_path),
    "paired_summary_test_only": str(paired_summary_path),
    "paired_rows_test_only": str(paired_rows_path) if not paired_rows.empty else None,
    "tables_dir": str(tables_dir),
    "plots_dir": str(plots_dir),
    "models_dir": str(models_dir),
}
write_json(artifact_manifest, output_dir / "analysis_artifact_manifest_graph_cold_od_ablation_v2_controls.json")

print("Saved v2 control ablation artifacts to:", output_dir)
print(json.dumps(artifact_manifest, indent=2, default=json_default))

Saved v2 control ablation artifacts to: E:\NetworkOptimization\pythonProject1\Data\10_experiments\graph_cold_od_ablation_v2_controls_notebook\east_plus_gulf
{
  "ablation_by_seed_predictions": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\graph_cold_od_ablation_v2_controls_notebook\\east_plus_gulf\\predictions_graph_cold_od_ablation_v2_controls_val_test_by_seed.parquet",
  "ablation_seed_ensemble_predictions": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\graph_cold_od_ablation_v2_controls_notebook\\east_plus_gulf\\predictions_graph_cold_od_ablation_v2_controls_val_test_seed_ensemble.parquet",
  "combined_predictions": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\graph_cold_od_ablation_v2_controls_notebook\\east_plus_gulf\\combined_predictions_graph_cold_od_ablation_v2_controls_val_test.parquet",
  "leaderboard_test": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\graph_cold_od_ablation_v2_controls_notebook\\east_p

## 27. Compact report without optional tabulate dependency

This report uses `DataFrame.to_string()` instead of `DataFrame.to_markdown()`, so it does not require the optional `tabulate` package.

In [27]:
def dataframe_to_text_table(frame: pd.DataFrame, columns: Sequence[str], max_rows: int = 10) -> str:
    """Convert a dataframe slice to a plain text table without optional dependencies."""
    if frame.empty:
        return "No rows available."
    cols = [column for column in columns if column in frame.columns]
    return frame[cols].head(max_rows).to_string(index=False)


report_lines: List[str] = []
report_lines.append("# Graph Cold-OD Ablation v2 Controls Report")
report_lines.append("")
report_lines.append(f"Output directory: {output_dir}")
report_lines.append(f"Enabled ablations: {list(cfg.enabled_ablation_names)}")
report_lines.append(f"Graph shuffle seeds: {list(cfg.graph_shuffle_seeds)}")
report_lines.append(f"Randomized edge-type seeds: {list(cfg.edge_type_randomization_seeds)}")
report_lines.append(f"Training seeds: {list(cfg.seeds)}")
report_lines.append(f"Split counts: {df['cold_split'].value_counts().to_dict()}")
report_lines.append("")
report_lines.append("## Top test leaderboard rows")
report_lines.append(dataframe_to_text_table(
    leaderboard_test,
    ["rank", "source", "model", "model_type", "node_feature_variant", "checkpoint_metric", "seed", "pinball_mean", "weighted_pinball_mean", "mae_q75", "iqr_mae", "stress_top10_mae_q75"],
    max_rows=25,
))
report_lines.append("")
report_lines.append("## Seed robustness summary")
report_lines.append(dataframe_to_text_table(
    seed_summary,
    ["source", "model", "model_type", "node_feature_variant", "perturbation_seed", "checkpoint_metric", "n_seeds", "pinball_mean_mean", "pinball_mean_std", "mae_q75_mean", "mae_q75_std", "iqr_mae_mean", "iqr_mae_std"],
    max_rows=40,
))
report_lines.append("")
report_lines.append("## Paired comparisons")
report_lines.append(dataframe_to_text_table(
    paired_summary,
    ["reference_name", "candidate_source", "candidate_model", "candidate_checkpoint_metric", "candidate_seed", "n_rows", "mean_delta_pinball_mean_row", "win_rate_pinball_mean_row", "mean_delta_abs_error_q75", "win_rate_abs_error_q75"],
    max_rows=50,
))
report_lines.append("")
report_lines.append("## Interpretation guide")
report_lines.append("- If node_features_only_edge_decoder is close to hgt_self_loop_only, node features rather than HGT self-loop message passing are the key source.")
report_lines.append("- If node_id_embedding_only is strong, zone identity memorization is a strong shortcut and must be reported carefully.")
report_lines.append("- If hgt_no_zone_numeric_no_degree_features degrades performance, zone-code or train-degree shortcuts are important.")
report_lines.append("- If hgt_constant_node_features_full_graph is strong relative to MLP, topology/message passing contributes beyond node attributes.")
report_lines.append("- If hgt_train_od_edges_removed_but_demand_kept is worse than full, observed train-OD transfer matters.")
report_lines.append("- If mode-specific demand variants differ, the corresponding freight mode context carries transferable signal.")
report_lines.append("- If several shuffled graph replicas degrade performance consistently, real FAF connectivity matters.")
report_lines.append("- If several randomized edge-type replicas degrade performance consistently, typed relation labels matter.")

report_path = reports_dir / "graph_cold_od_ablation_v2_controls_report.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print("Report written to:", report_path)
print("\n".join(report_lines[:25]))


Report written to: E:\NetworkOptimization\pythonProject1\Data\10_experiments\graph_cold_od_ablation_v2_controls_notebook\east_plus_gulf\reports\graph_cold_od_ablation_v2_controls_report.md
# Graph Cold-OD Ablation v2 Controls Report

Output directory: E:\NetworkOptimization\pythonProject1\Data\10_experiments\graph_cold_od_ablation_v2_controls_notebook\east_plus_gulf
Enabled ablations: ['hgt_full_faf_zone_graph', 'graphsage_same_edges', 'mlp_same_features_no_graph', 'node_features_only_edge_decoder', 'node_id_embedding_only', 'hgt_self_loop_only', 'hgt_no_zone_numeric_no_degree_features', 'hgt_constant_node_features_full_graph', 'hgt_train_od_only', 'hgt_train_od_edges_removed_but_demand_kept', 'hgt_demand_edges_only', 'hgt_truck_demand_only', 'hgt_rail_demand_only', 'hgt_multimodal_demand_only', 'hgt_truck_plus_rail_demand', 'hgt_truck_plus_multimodal_demand', 'hgt_rail_plus_multimodal_demand', 'hgt_shuffled_graph_edges_s1', 'hgt_shuffled_graph_edges_s2', 'hgt_shuffled_graph_edges_s3',